# Cuaderno 20 · Motor de inyección (enmienda D)

Correr en orden, una vez cada celda: M1 a M8 (M8 es el control contra H1, unos 2 minutos por semilla). Solo si M8 dice **REPRODUCE H1: sí**, seguir con M9 (E0), M10 (E3b), M11 (contrastes de E0 y E3b), M12 a M14 (E1) y M15 a M17 (E2). M15 verifica las celdas anteriores y arma los tipos de E2 sin ajustar modelos; M16 corre E2 (unas 2 horas, guarda tras cada semilla); M17 arma las tablas. No usar "Ejecutar todo". Guardar el cuaderno (Ctrl+S) antes de M1 y antes de M15.

In [ ]:
# =====================================================================================
# CUADERNO 20 (MOTOR) · CELDA M1. Configuración y verificaciones
#   Motor de inyección de la enmienda D (sección 5.1): el mismo de H1 (cuaderno 17), generalizado
#   al año de inyección, al tipo de anomalía y a la representación del precio.
#   Análisis que corre este cuaderno: control de reproducción de H1, E0 y E3b. Exploratorios.
#   Antes de correr: guardar el cuaderno (Ctrl+S); la celda toma la huella de su código desde el archivo.
#   No carga planA_validacion_sri ni la muestra de H3; no imprime RUC ni procesos.
# =====================================================================================
import os, time, json, math, hashlib, platform, warnings, unicodedata
from datetime import datetime
import numpy as np
import pandas as pd
import duckdb
import sklearn
from scipy.stats import spearmanr, norm
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.neural_network import MLPRegressor
from sklearn.exceptions import ConvergenceWarning

SALIDA_TXT, CELDA_ACTUAL = {}, "M1"
def say(*args):
    texto = " ".join(str(a) for a in args)
    print(texto)
    SALIDA_TXT.setdefault(CELDA_ACTUAL, []).append(texto)

def empezar(nombre):
    global CELDA_ACTUAL
    CELDA_ACTUAL = nombre
    SALIDA_TXT[nombre] = []

# --- rutas ---
SERCOP   = r"C:\SERCOP_FINAL"
BASE     = os.path.join(SERCOP, "plan_A")
SALIDA   = os.path.join(SERCOP, "modelos")
PREREG   = os.path.join(SERCOP, "preregistro")
CUADERNO = os.path.join(BASE, "cuaderno20_motor.ipynb")
ENM_D    = os.path.join(SERCOP, "enmiendas", "enmienda_2026-11-10_d.json")
H1_DIR   = os.path.join(SALIDA, "20261016_1207_h1final")
HUELLA_D_ESPERADA = "55b01c16035d3b2b"

# --- parámetros del preregistro (iguales al cuaderno 17; no se cambian) ---
MODO = "COMPLETO"
REPRESENTACION = "normal_sup"
SEMILLA = 20261001
IF_ARBOLES, IF_SUBMUESTRA, LOF_VECINOS = 100, 256, 20
CORTE_TOP, MIN_PATRON, ANIO_INICIO = 0.01, 1000, 2016
MIN_GRUPO, REF_MAX, AE_MAX_ENTRENO, HUERFANO_MAX = 100, 50_000, 50_000, 10_000

# --- parámetros de la enmienda D (5.1, 5.2, 5.5 y 6) ---
ANIOS_CONFIRMACION = {2022: 2021, 2019: 2018}     # año: reemplazo si tiene menos de 4 grupos con 1.000 procesos
SEMILLAS_EXPLORACION = list(range(1, 21))         # 2024
SEMILLAS_CONFIRMACION = list(range(21, 41))       # 2022 y 2019
B_REMUESTREO = 2000
MIN_CELDA_FINA = 30                               # E3b: celda fina con menos procesos conserva la referencia vigente

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
warnings.filterwarnings("ignore", category=RuntimeWarning, message="Mean of empty slice")
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
T_INICIO = time.time()
ALERTAS = []

def sha_completa(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

# preregistro firmado y nada cambiado desde la firma (como en el cuaderno 17)
FIRMA = json.load(open(os.path.join(PREREG, "preregistro_firma.json"), encoding="utf-8"))
if sha_completa(os.path.join(PREREG, FIRMA["documento"])) != FIRMA["huella_documento"]:
    raise SystemExit("El preregistro cambió después de firmarlo: para y avisa.")
for rel, h in FIRMA["huellas_citadas"].items():
    if sha_completa(os.path.join(SERCOP, rel)) != h:
        raise SystemExit(f"{rel} cambió después de la firma: para y avisa.")
HUELLA_D = sha_completa(ENM_D)
if not HUELLA_D.startswith(HUELLA_D_ESPERADA):
    raise SystemExit("La enmienda D no es la registrada: para y avisa.")

# huella del código de este cuaderno (celdas cuyo título empieza con "# CUADERNO 20 (MOTOR)")
def huella_codigo_cuaderno():
    nb = json.load(open(CUADERNO, encoding="utf-8"))
    partes = []
    for c in nb["cells"]:
        if c["cell_type"] != "code":
            continue
        src = c["source"] if isinstance(c["source"], str) else "".join(c["source"])
        if any(l.startswith("# CUADERNO 20 (MOTOR)") for l in src.splitlines()[:3]):
            partes.append(src)
    return hashlib.sha256("\n# ----- fin de celda -----\n".join(partes).encode("utf-8")).hexdigest(), len(partes)
if not os.path.exists(CUADERNO):
    raise SystemExit(f"No encuentro {CUADERNO}: guarda el cuaderno con ese nombre.")
HUELLA_CODIGO, N_CELDAS = huella_codigo_cuaderno()

VERSIONES = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                 sklearn=sklearn.__version__, duckdb=duckdb.__version__)
say(f"Preregistro verificado ({FIRMA['huella_documento'][:16]}) | enmienda D {HUELLA_D[:16]} | "
    f"código del motor {HUELLA_CODIGO[:16]} ({N_CELDAS} celdas)")
say(" | ".join(f"{k} {v}" for k, v in VERSIONES.items()) + f" | núcleos {os.cpu_count()}")

In [ ]:
# =====================================================================================
# CUADERNO 20 (MOTOR) · CELDA M2. Carga de la base y grupos comparables (igual a la celda 2 del cuaderno 17)
#          (NO se carga planA_validacion_sri: la marca del SRI se usa una sola vez, al final)
# =====================================================================================
empezar("M2")
def ruta(nombre):
    return os.path.join(BASE, nombre).replace("\\", "/")

def huella(path, n=16):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()[:n]

def a_float(s):
    """Convierte booleanos, enteros o nulos a float (nulo -> NaN) sin errores de tipo."""
    try:
        return pd.Series(pd.array(s, dtype="Float64").to_numpy(dtype="float64", na_value=np.nan), index=s.index)
    except Exception:
        return pd.to_numeric(s.astype("object"), errors="coerce").astype("float64")

def guardar_parquet(tabla, nombre):
    """Guarda en la carpeta de la corrida; si pandas no tiene motor de parquet, usa DuckDB."""
    destino = os.path.join(DIR_RUN, nombre)
    try:
        tabla.to_parquet(destino)
    except ImportError:
        c = duckdb.connect()
        c.register("t_guardar", tabla.reset_index())
        c.execute(f"COPY t_guardar TO '{destino.replace(chr(92), '/')}' (FORMAT PARQUET)")
        c.close()

def sin_tildes(texto):
    t = unicodedata.normalize("NFKD", str(texto))
    return "".join(c for c in t if not unicodedata.combining(c)).lower().strip()

t_celda = time.time()
IX = ruta("planA_indicadores.parquet")
PM = ruta("planA_procesos_modelo.parquet")
for p in (IX, PM):
    if not os.path.exists(p):
        raise SystemExit(f"No encuentro {p}. Revisa BASE en la celda 1.")
HUELLA_IX, HUELLA_PM = huella(IX), huella(PM)

con = duckdb.connect()
ix = con.execute(f"SELECT * FROM read_parquet('{IX}')").df()
cols_pm = [r[0] for r in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{PM}')").fetchall()]
pedir = ["ocid"] + [c for c in ["familia", "fecha_inicio_tender", "monto_final", "entidad_ruc", "proveedor_ruc_final"]
                    if c in cols_pm and c not in ix.columns]
pm = con.execute(f"SELECT {', '.join(pedir)} FROM read_parquet('{PM}')").df()
say(f"planA_indicadores: {ix.shape[0]:,} filas x {ix.shape[1]} columnas | huella {HUELLA_IX}")
say(f"planA_procesos_modelo: {pm.shape[0]:,} filas; se toman {pedir[1:]} | huella {HUELLA_PM}")
say("\nColumnas de planA_indicadores:")
say(", ".join(ix.columns))

df = ix.merge(pm, on="ocid", how="left")
if len(df) != len(ix) or df["ocid"].duplicated().any():
    raise SystemExit("La unión por ocid cambió el número de filas: revisar antes de seguir.")
say(f"\nProcesos de indicadores sin pareja en procesos_modelo: {(~ix['ocid'].isin(pm['ocid'])).sum():,}")
if "familia" not in df.columns:
    raise SystemExit("No encuentro la columna 'familia' en ninguno de los dos archivos.")

# fecha del proceso = inicio del período de ofertas
col_fecha = next((c for c in ["fecha_inicio_tender", "fecha_proceso"] if c in df.columns), None)
if col_fecha is None:
    raise SystemExit("No encuentro la fecha del proceso (fecha_inicio_tender).")
fecha = pd.to_datetime(df[col_fecha], errors="coerce")
try:
    if fecha.dt.tz is not None:
        fecha = fecha.dt.tz_convert("America/Guayaquil").dt.tz_localize(None)
except AttributeError:
    pass
say(f"Fecha del proceso tomada de '{col_fecha}'; procesos sin fecha: {int(fecha.isna().sum()):,}")
df["anio_proc"] = fecha.dt.year
df["periodo_calc"] = np.where(fecha >= pd.Timestamp("2025-10-07"), "post", "pre")
if "periodo" in df.columns:
    coinc = (df["periodo"].astype(str).str.lower().str.startswith("post") == (df["periodo_calc"] == "post")).mean()
    say(f"Coincidencia entre 'periodo' de la base y el corte por fecha: {coinc:.4%}")
df["grupo"] = df["familia"].astype(str) + " | " + df["periodo_calc"]
if "monto_final" not in df.columns:
    df["monto_final"] = np.nan
df["monto_final"] = a_float(df["monto_final"])

# Control de calibración sobre la base completa (antes de quitar 2015 y de muestrear):
# cada percentil de la base debe tener media 0,500 dentro de su grupo
say("\nControl de calibración de la base (media de cada percentil dentro de su grupo; debe dar 0,500):")
filas = []
for c in ["pct_precio_alto", "pct_monto_capital", "pct_proveedor_nuevo", "pct_plazo_corto", "pct_firma_rapida",
          "pct_fraccionamiento_pos", "pct_cuota_pos", "pct_recurrencia_pos"]:
    if c in df.columns:
        s_ = a_float(df[c])
        g_ = s_.groupby(df["grupo"])
        m = g_.mean()[g_.count() >= 200]                 # solo grupos con al menos 200 valores
        filas.append(dict(columna=c, media_min=m.min(), media_max=m.max(),
                          grupo_mas_lejos=(m - 0.5).abs().idxmax() if m.notna().any() else "-"))
cal = pd.DataFrame(filas)
say(cal.round(4).to_string(index=False))
lejos = cal[(cal["media_min"] < 0.495) | (cal["media_max"] > 0.505)]
if len(lejos):
    ALERTAS.append("Percentiles de la base con media lejos de 0,5 en algún grupo: " + ", ".join(lejos["columna"]))

# Debutante con ventana de 12 meses (enmienda): sin procesos del proveedor en los 365 días anteriores.
# Sale de la misma columna de historial que usan cuota y recurrencia (verificada en la celda 0).
if "ind_procesos_proveedor_12m" not in df.columns:
    raise SystemExit("No encuentro ind_procesos_proveedor_12m: corre primero la celda 0 y pégame su salida.")
n12 = a_float(df["ind_procesos_proveedor_12m"])
df["ind_proveedor_debutante_365"] = np.where(n12.isna(), np.nan, (n12 == 0).astype(float))

# 2015 queda solo como historial (enmienda): se retira de lo que se puntúa
DF_TODO = df.copy()     # base completa (con 2015): de aquí salen las distribuciones del recálculo
quitados = df.loc[df["anio_proc"] < ANIO_INICIO].groupby("grupo").size()
df = df.loc[df["anio_proc"] >= ANIO_INICIO].reset_index(drop=True)
say(f"\nProcesos anteriores a {ANIO_INICIO} retirados (siguen sirviendo de historial): {int(quitados.sum()):,}")
say(f"Procesos puntuables: {len(df):,}")

TAM_REAL = df.groupby("grupo").size()            # tamaño de cada grupo puntuable en la base completa
if MODO == "RAPIDO":
    df = (df.sample(frac=1.0, random_state=SEMILLA)
            .groupby("grupo", sort=False).head(RAPIDO_MAX_GRUPO)
            .sort_values("ocid").reset_index(drop=True))
    say(f"\nModo RAPIDO: se trabaja con {len(df):,} procesos (hasta {RAPIDO_MAX_GRUPO:,} por grupo).")

tab = (df.groupby("grupo").agg(procesos=("ocid", "size"),
                               de_2023=("anio_proc", lambda s: int((s == 2023).sum())),
                               anio_min=("anio_proc", "min"), anio_max=("anio_proc", "max"),
                               millones=("monto_final", lambda s: round(s.sum() / 1e6, 1))))
tab.insert(0, "en_la_base", TAM_REAL)
tab.insert(1, f"quitados_antes_{ANIO_INICIO}", quitados.reindex(tab.index).fillna(0).astype(int))
tab = tab.sort_values("en_la_base", ascending=False)
say("\nGrupos comparables (familia | período):")
say(tab.to_string())
say("\nRecordatorio: planA_validacion_sri NO se cargó. Ninguna cifra de esta corrida usa la marca del SRI.")
say(f"[celda 2: {time.time() - t_celda:.0f} s]")

In [ ]:
# =====================================================================================
# CUADERNO 20 (MOTOR) · CELDA M3. Codificación de las variables (igual a la celda 3 del cuaderno 17)
#   rango      -> percentil del grupo (ya viene en la base); si falta el dato queda vacío (NaN)
#   rango_pos  -> percentil entre los procesos donde el fenómeno ocurre; 0 cuando no ocurre
#                 (cero real: no hubo historial o no hubo recompra), nunca un relleno
#   binaria    -> 0/1; vacío si falta
#   dif        -> diferencia con la mediana de oferentes del grupo, llevada a 0-1 dentro del grupo;
#                 1 cuando hubo un solo oferente (el extremo real de la escala)
# =====================================================================================
empezar("M3")
t_celda = time.time()

def pct_mitad_salto(s, grupos):
    """Percentil a mitad de salto dentro de cada grupo (los vacíos quedan vacíos)."""
    r = s.groupby(grupos).rank(method="average")
    n = s.notna().groupby(grupos).transform("sum")
    return (r - 0.5) / n

VARIABLES = {
    # nombre             columna en la base                  tipo         entra a la línea base
    "precio_alto":       ("pct_precio_alto",                 "rango",     True),
    "monto_capital":     ("pct_monto_capital",               "rango",     True),
    "proveedor_nuevo":   ("pct_proveedor_nuevo",             "rango",     True),
    "plazo_corto":       ("pct_plazo_corto",                 "rango",     True),
    "firma_rapida":      ("pct_firma_rapida",                "rango",     True),
    "recompra_30d":      ("pct_fraccionamiento_pos",         "rango_pos", True),
    "cuota_entidad":     ("pct_cuota_pos",                   "rango_pos", True),
    "recurrencia_par":   ("pct_recurrencia_pos",             "rango_pos", True),
    "oferente_unico":    ("ind_oferente_unico",              "binaria",   True),
    "poca_concurrencia": ("dif_concurrencia_grupo",          "dif",       True),
    # primera_vez_ent sale de los modelos: con la ventana de 12 meses coincide exactamente con
    # "recurrencia = 0", así que no aporta información nueva (enmienda del 16-oct-2026)
    "debutante":         ("ind_proveedor_debutante_365",     "binaria",   True),
}
# columnas de origen de las variables en dos partes: la que dice si el fenómeno ocurre (> 0) y la
# que mide su tamaño (solo se usa si hubiera que reconstruir el percentil)
ORIGEN_POS = {"recompra_30d":    ["ind_fraccionamiento_marca", "ind_fraccionamiento_monto_30d"],
              "cuota_entidad":   ["ind_cuota_prov_en_entidad_12m"],
              "recurrencia_par": ["ind_recurrencia_par_12m"]}
MAGNITUD_POS = {"recompra_30d":    ["ind_fraccionamiento_monto_30d"],
                "cuota_entidad":   ["ind_cuota_prov_en_entidad_12m"],
                "recurrencia_par": ["ind_recurrencia_par_12m"]}

def primera(cols):
    return next((c for c in cols if c in df.columns), None)

# Si una columna no existe con el nombre esperado, se reconstruye desde su origen (y se avisa)
RECONSTRUIDAS = []
for v, origenes in MAGNITUD_POS.items():
    col, origen = VARIABLES[v][0], primera(origenes)
    if col not in df.columns and origen is not None:
        mag = a_float(df[origen])
        df[col] = pct_mitad_salto(mag.where(mag > 0), df["grupo"])
        RECONSTRUIDAS.append(f"{v} desde {origen}")
col_nof = primera(["ind_n_oferentes", "n_oferentes"])
if col_nof is not None:
    nof = a_float(df[col_nof])
    if "ind_oferente_unico" not in df.columns:
        df["ind_oferente_unico"] = np.where(nof.isna(), np.nan, (nof == 1).astype(float))
        RECONSTRUIDAS.append(f"oferente_unico desde {col_nof}")
    if "dif_concurrencia_grupo" not in df.columns:
        df["dif_concurrencia_grupo"] = (nof.groupby(df["grupo"]).transform("median") - nof).where(nof >= 2)
        RECONSTRUIDAS.append(f"poca_concurrencia desde {col_nof}")

X = pd.DataFrame(index=df.index)
say("Variables del modelo y columna de origen:")
for v, (col, tipo, _) in list(VARIABLES.items()):
    if col not in df.columns:
        say(f"  {v:<18} NO ENCONTRADA ({col}) -> queda fuera")
        ALERTAS.append(f"Variable {v} no encontrada en la base (columna {col}).")
        VARIABLES.pop(v)
        continue
    s = a_float(df[col])
    nota = ""
    if tipo == "rango_pos":
        origen = primera(ORIGEN_POS[v])
        if origen is not None:
            o = a_float(df[origen])
            ocurre, no_ocurre = o > 0, o <= 0
            mal = int((s.isna() & ocurre).sum() + ((s > 0) & no_ocurre).sum())
            nota = f"control de ceros con {origen}: {mal:,} filas incoherentes" + ("  <- REVISAR" if mal else " (ok)")
            if mal:
                ALERTAS.append(f"{v}: {mal:,} filas donde el percentil y su origen no coinciden.")
            s = s.where(~(no_ocurre & s.isna()), 0.0)
        else:
            nota = "sin columna de origen: vacío se toma como 'no ocurre'"
            s = s.fillna(0.0)
    X[v] = s
    say(f"  {v:<18} {col:<32} {tipo:<10} cobertura {100 * s.notna().mean():5.1f} %   {nota}")
if RECONSTRUIDAS:
    say("Reconstruidas porque la columna esperada no existe:", "; ".join(RECONSTRUIDAS))
    ALERTAS.append("Variables reconstruidas desde su origen: " + "; ".join(RECONSTRUIDAS))
if X.shape[1] < 6:
    raise SystemExit("Faltan demasiadas variables. Pega en el chat la lista de columnas de la celda 2.")

# poca_concurrencia: escala 0-1 dentro de cada grupo, 1 = un solo oferente
if "poca_concurrencia" in X.columns:
    pc = X["poca_concurrencia"].copy()
    for g, idx in df.groupby("grupo").groups.items():
        vals = pc.loc[idx].dropna()
        if len(vals) >= 30:
            lo, hi = np.nanpercentile(vals, 1), np.nanpercentile(vals, 99)
            if hi > lo:
                pc.loc[idx] = ((pc.loc[idx].clip(lo, hi) - lo) / (hi - lo))
            else:
                pc.loc[idx] = np.where(pc.loc[idx].notna(), 0.5, np.nan)
        else:
            pc.loc[idx] = np.where(pc.loc[idx].notna(), 0.5, np.nan)
    if "oferente_unico" in X.columns:
        pc = pc.where(~(X["oferente_unico"] == 1), 1.0)
        pc = pc.where(~X["oferente_unico"].isna(), np.nan)
    X["poca_concurrencia"] = pc

ORIENTADAS = [v for v, (_, _, base) in VARIABLES.items() if base]
TIPO = {v: t for v, (_, t, _) in VARIABLES.items()}
CORTO = {"precio_alto": "precio", "monto_capital": "capital", "proveedor_nuevo": "prov_nuevo",
         "plazo_corto": "plazo", "firma_rapida": "firma", "recompra_30d": "recompra",
         "cuota_entidad": "cuota", "recurrencia_par": "recurr", "oferente_unico": "of_unico",
         "poca_concurrencia": "poca_conc", "primera_vez_ent": "primera_vez", "debutante": "debutante"}

# control de orientación (alto = anómalo); la calibración ya se revisó en la celda 2
def control_orientacion(var, origen, signo):
    if var in X.columns and origen in df.columns:
        a, b = X[var], a_float(df[origen])
        ok = a.notna() & b.notna() & ((a > 0) if TIPO[var] == "rango_pos" else True)
        if ok.sum() > 100:
            r = spearmanr(a[ok], b[ok]).correlation
            estado = "ok" if np.sign(r) == signo else "REVISAR ORIENTACIÓN"
            if estado != "ok":
                ALERTAS.append(f"Orientación dudosa: {var} frente a {origen} (rho {r:+.2f}).")
            say(f"  {var:<18} vs {origen:<32} rho = {r:+.3f}  (esperado {'+' if signo > 0 else '-'})  {estado}")
say("\nControl de orientación (alto = más anómalo):")
control_orientacion("precio_alto", "ind_desv_precio_unit_z", +1)
control_orientacion("monto_capital", "ind_monto_sobre_capital", +1)
control_orientacion("proveedor_nuevo", "ind_antiguedad_dias", -1)
control_orientacion("plazo_corto", "ind_plazo_ofertas_dias", -1)
control_orientacion("firma_rapida", "ind_plazo_adj_firma_dias", -1)
control_orientacion("recompra_30d", "ind_fraccionamiento_monto_30d", +1)
control_orientacion("cuota_entidad", "ind_cuota_prov_en_entidad_12m", +1)
control_orientacion("recurrencia_par", "ind_recurrencia_par_12m", +1)
if col_nof is not None:
    control_orientacion("poca_concurrencia", col_nof, -1)

cob = (X.notna().groupby(df["grupo"]).mean() * 100).round(1)
cob.insert(0, "procesos", df.groupby("grupo").size())
say("\nCobertura de cada variable por grupo (% de procesos con dato):")
say(cob.sort_values("procesos", ascending=False).rename(columns=CORTO).to_string())

# (el cuaderno 17 guardaba aquí X y las llaves; el motor no las guarda)
say(f"[celda 3: {time.time() - t_celda:.0f} s]")

In [ ]:
# =====================================================================================
# CUADERNO 20 (MOTOR) · CELDA M4. Patrones de disponibilidad (igual a la celda 4 del cuaderno 17)
#   - Se retiran del grupo las variables casi ausentes (<1 %) o constantes (>= 99,5 % un valor).
#   - Cada patrón con >= MIN_PATRON procesos tiene modelo propio ("grande").
#   - Un patrón menor se puntúa con el modelo grande que usa solo variables que él sí tiene;
#     si no hay ninguno, recibe un modelo propio ajustado sin la variable que le falta ("pequeño").
# =====================================================================================
empezar("M4")
def variables_del_grupo(Xg):
    usar, fuera = [], []
    for v in Xg.columns:
        s = Xg[v].dropna()
        if len(s) < max(30, 0.01 * len(Xg)):
            fuera.append(f"{v} (sin dato)")
        elif s.value_counts(normalize=True).iloc[0] >= 0.995:
            fuera.append(f"{v} (constante)")
        else:
            usar.append(v)
    return usar, fuera

def plan_de_modelos(Xg, usar):
    disp = Xg[usar].notna().to_numpy()
    codigo = (disp * (1 << np.arange(len(usar)))).sum(axis=1)
    texto = {c: "".join("1" if (c >> j) & 1 else "0" for j in range(len(usar))) for c in np.unique(codigo)}
    clave = pd.Series([texto[c] for c in codigo], index=Xg.index)
    conteo = clave.value_counts()
    grandes = [k for k, n in conteo.items() if n >= MIN_PATRON] or [conteo.index[0]]
    def vars_de(k): return [u for u, b in zip(usar, k) if b == "1"]
    asign = {}
    for k in conteo.index:
        if k in grandes:
            asign[k] = k
            continue
        mias = set(vars_de(k))
        cand = [g for g in grandes if set(vars_de(g)) <= mias]
        asign[k] = max(cand, key=lambda g: (len(vars_de(g)), conteo[g])) if cand else k
    modelos = sorted(set(asign.values()), key=lambda k: -conteo.get(k, 0))
    return clave, conteo, asign, {m: vars_de(m) for m in modelos}, set(grandes)

PLAN, NO_MODELADOS, filas = {}, {}, []
for g in df.groupby("grupo").size().sort_values(ascending=False).index:
    idx = df.index[df["grupo"] == g]
    Xg = X.loc[idx]
    usar, fuera = variables_del_grupo(Xg)
    if len(idx) < MIN_GRUPO or len(usar) < 2:
        NO_MODELADOS[g] = f"{len(idx)} procesos, {len(usar)} variables útiles"
        continue
    clave, conteo, asign, vars_modelo, grandes = plan_de_modelos(Xg, usar)
    PLAN[g] = dict(idx=idx, usar=usar, clave=clave, asign=asign, vars_modelo=vars_modelo, grandes=grandes)
    modelo_de = clave.map(asign)
    filas.append(dict(grupo=g, procesos=len(idx), variables=len(usar), patrones=len(conteo),
                      modelos_grandes=sum(m in grandes for m in vars_modelo),
                      modelos_pequenos=sum(m not in grandes for m in vars_modelo),
                      pct_en_modelo_principal=round(100 * (modelo_de == conteo.index[0]).mean(), 1),
                      pct_en_modelos_pequenos=round(100 * (~modelo_de.isin(grandes)).mean(), 2),
                      vars_modelo_principal=conteo.index[0].count("1"),
                      fuera=", ".join(fuera)))
say(pd.DataFrame(filas).to_string(index=False))
if NO_MODELADOS:
    say("\nGrupos que no se modelan:", NO_MODELADOS)

In [ ]:
# =====================================================================================
# CUADERNO 20 (MOTOR) · CELDA M5. Funciones de los modelos (iguales a la celda 5 del cuaderno 17)
# =====================================================================================
empezar("M5")
def pct_vs_ref(ref, x):
    """Percentil a mitad de salto de cada valor de x frente a la distribución de referencia ref."""
    ref = np.sort(np.asarray(ref, dtype=float))
    lo = np.searchsorted(ref, x, side="left")
    hi = np.searchsorted(ref, x, side="right")
    return (lo + 0.5 * (hi - lo)) / len(ref)

def ajustar_un_modelo(Xtr, Xsc, semilla, ref_max, ae_max):
    """Entrena los tres detectores con Xtr (procesos que tienen todas las variables del modelo)
    y puntúa Xsc. Devuelve percentiles (alto = más anómalo) frente a una referencia tomada de Xtr."""
    rng = np.random.default_rng(semilla)
    n, d = Xtr.shape
    R = Xtr[rng.choice(n, size=min(n, ref_max), replace=False)]
    out, tiempos = {}, {}
    # Isolation Forest
    t = time.time()
    iso = IsolationForest(n_estimators=IF_ARBOLES, max_samples=min(IF_SUBMUESTRA, n),
                          random_state=semilla, n_jobs=-1).fit(Xtr)
    out["if"] = pct_vs_ref(-iso.score_samples(R), -iso.score_samples(Xsc))
    tiempos["if"] = time.time() - t
    # Local Outlier Factor (modo novedad). Se ajusta sobre las filas distintas de la referencia.
    # Un proceso idéntico a una fila de la referencia recibe el LOF de esa fila, calculado sin
    # contarse a sí misma; así todos los procesos se miden con la misma regla.
    t = time.time()
    U, inv = np.unique(R, axis=0, return_inverse=True)
    inv = np.asarray(inv).ravel()
    if len(U) > LOF_VECINOS + 5:
        lof = LocalOutlierFactor(n_neighbors=LOF_VECINOS, novelty=True, n_jobs=-1).fit(U)
        lof_u = -lof.negative_outlier_factor_
        _, inv_all = np.unique(np.vstack([U, Xsc]), axis=0, return_inverse=True)
        inv_all = np.asarray(inv_all).ravel()
        pos = np.full(inv_all.max() + 1, -1)
        pos[inv_all[:len(U)]] = np.arange(len(U))
        m = pos[inv_all[len(U):]]
        s = np.empty(len(Xsc))
        s[m >= 0] = lof_u[m[m >= 0]]
        if (m < 0).any():
            s[m < 0] = -lof.score_samples(Xsc[m < 0])
        out["lof"] = pct_vs_ref(lof_u[inv], s)
    else:
        out["lof"] = np.full(len(Xsc), np.nan)
    tiempos["lof"] = time.time() - t
    # Autoencoder: red simétrica con capa intermedia de la mitad de las variables
    t = time.time()
    if d >= 3:
        tr_i = rng.choice(n, size=min(n, ae_max), replace=False)
        ae = MLPRegressor(hidden_layer_sizes=(max(2, d // 2),), activation="relu", solver="adam",
                          alpha=1e-4, batch_size=256, learning_rate_init=1e-3, max_iter=200,
                          early_stopping=True, validation_fraction=0.1, n_iter_no_change=10,
                          random_state=semilla)
        ae.fit(Xtr[tr_i], Xtr[tr_i])
        err_ref = ((ae.predict(R) - R) ** 2).mean(axis=1)
        err_sc = ((ae.predict(Xsc) - Xsc) ** 2).mean(axis=1)
        out["ae"] = pct_vs_ref(err_ref, err_sc)
    else:
        out["ae"] = np.full(len(Xsc), np.nan)
    tiempos["ae"] = time.time() - t
    return out, tiempos

def top_k(valores, frac, semilla):
    """Marca el frac superior (exactamente k procesos con puntaje; empates al azar con semilla fija)."""
    v = np.asarray(valores, dtype=float)
    fin = np.isfinite(v)
    marca = np.zeros(len(v), dtype=bool)
    if fin.sum() == 0:
        return marca
    k = max(1, int(math.ceil(frac * fin.sum())))
    desempate = np.random.default_rng(semilla).random(len(v))
    orden = np.lexsort((desempate, -np.where(fin, v, -np.inf)))
    marca[orden[:k]] = True
    return marca

def rareza(Xg_df):
    """Percentil a mitad de salto de cada variable dentro del conjunto (alto = valor más alto)."""
    return (Xg_df.rank(method="average") - 0.5) / Xg_df.notna().sum()

def entrada_modelo(Xg_df):
    """Lo que reciben los modelos. 'percentil': los valores tal cual. 'normal_sup': la rareza de cada
    valor dentro del grupo en escala normal, con la mitad baja (sin riesgo) llevada a 0."""
    if REPRESENTACION == "normal_sup":
        z = norm.ppf(rareza(Xg_df).clip(1e-9, 1 - 1e-9).to_numpy(dtype=float))
        return pd.DataFrame(np.maximum(z, 0.0), index=Xg_df.index, columns=Xg_df.columns).where(Xg_df.notna())
    return Xg_df

def puntuar_grupo(plan, Xg_df, semilla, verbose=False):
    """Ajusta todos los modelos de un grupo. Devuelve percentiles por proceso, tiempos y rareza."""
    res = pd.DataFrame(index=Xg_df.index)
    Xm = entrada_modelo(Xg_df[plan["usar"]])
    for m in ["if", "lof", "ae"]:
        res[m] = np.nan
    res["modelo"] = ""
    t_mod = {"if": 0.0, "lof": 0.0, "ae": 0.0}
    clave = plan["clave"].loc[Xg_df.index]
    modelo_de = clave.map(plan["asign"])
    for j, (mk, vs) in enumerate(plan["vars_modelo"].items(), start=1):
        tm = time.time()
        destino = modelo_de.index[modelo_de == mk]
        if len(destino) == 0 or len(vs) == 0:
            continue
        tr_mask = Xg_df[vs].notna().all(axis=1)
        if tr_mask.sum() < 30:
            continue
        tope = (REF_MAX, AE_MAX_ENTRENO) if mk in plan["grandes"] else (HUERFANO_MAX, HUERFANO_MAX)
        out, tt = ajustar_un_modelo(Xm.loc[tr_mask, vs].to_numpy(dtype=float),
                                    Xm.loc[destino, vs].to_numpy(dtype=float), semilla, *tope)
        for m in out:
            res.loc[destino, m] = out[m]
            t_mod[m] += tt[m]
        res.loc[destino, "modelo"] = mk
        if verbose:
            say(f"      modelo {j}/{len(plan['vars_modelo'])}: {len(vs)} variables, entrena con "
                  f"{int(tr_mask.sum()):,}, puntúa {len(destino):,}  ({time.time() - tm:.0f} s)")
    res["ensamble"] = res[["if", "lof", "ae"]].mean(axis=1, skipna=True)
    rz = rareza(Xg_df[plan["usar"]])
    orient = [v for v in ORIENTADAS if v in plan["usar"]]
    res["base"] = rz[orient].mean(axis=1, skipna=True) if orient else np.nan
    res["n_vars_disp"] = Xg_df[plan["usar"]].notna().sum(axis=1)
    return res, t_mod, rz

def explicar(rz, cols, n=3):
    """Para cada fila, las n variables con valor más alto dentro del grupo."""
    a = rz[cols].to_numpy(dtype=float)
    a = np.where(np.isfinite(a), a, -1.0)
    orden = np.argsort(-a, axis=1)[:, :n]
    return [" | ".join(f"{cols[j]} {a[i, j]:.2f}" for j in orden[i] if a[i, j] >= 0) for i in range(len(a))]

def marcar_tops(res, semilla):
    for m in ["ensamble", "if", "lof", "ae", "base"]:
        res[f"top_{m}"] = top_k(res[m], CORTE_TOP, semilla)
    res["consenso"] = res[["top_if", "top_lof", "top_ae"]].sum(axis=1) >= 2
    return res
say("Funciones de los modelos cargadas.")

In [ ]:
# =====================================================================================
# CUADERNO 20 (MOTOR) · CELDA M6. Recetas de inyección (las de la celda 6 del cuaderno 17, para cualquier año)
#          su indicador con la misma regla de la base (la que verificó la celda R)
# =====================================================================================
empezar("M6")
t_celda = time.time()
REPRO = json.load(open(os.path.join(PREREG, "reproduccion_indicadores.json"), encoding="utf-8"))
if REPRO.get("veredicto") != "OK" or REPRO.get("huella_indicadores") != HUELLA_IX:
    raise SystemExit("La reproducción de la celda R no es OK o es de otra versión de la base.")
REGLAS = REPRO["percentiles"]

def num(s):
    return a_float(s)

def clave_grupo(tabla, agrupacion):
    if agrupacion == "familia|periodo":
        return tabla["familia"].astype(str) + " | " + tabla["periodo"].astype(str)
    return tabla["familia"].astype(str)

def valores_orientados(tabla, regla):
    x = num(tabla[regla["fuente"]])
    if regla["subconjunto"] == "positivos":
        x = x.where(x > 0)
    return regla["signo"] * x

# distribución original de cada fuente en la base completa (con 2015, como se construyó la base)
ORDENADOS = {}
for pct, regla in REGLAS.items():
    v = valores_orientados(DF_TODO, regla)
    k = clave_grupo(DF_TODO, regla["agrupacion"])
    ORDENADOS[pct] = {g: np.sort(s.dropna().to_numpy()) for g, s in v.groupby(k)}

def recalcular(pct, filas, nuevos_origen):
    """Nuevo percentil de las filas `filas` de df si su variable de origen pasa a `nuevos_origen`.
    Se compara con la distribución original del grupo, quitando el valor original del propio proceso."""
    regla = REGLAS[pct]
    sub = df.loc[filas]
    claves = clave_grupo(sub, regla["agrupacion"]).to_numpy()
    orig = valores_orientados(sub, regla).to_numpy()
    nuevo = regla["signo"] * np.asarray(nuevos_origen, dtype=float)
    salida = np.full(len(filas), np.nan)
    for g in np.unique(claves):
        m = claves == g
        arr = ORDENADOS[pct].get(g, np.array([]))
        v, o = nuevo[m], orig[m]
        debajo = np.searchsorted(arr, v, side="left").astype(float)
        iguales = np.searchsorted(arr, v, side="right") - debajo
        propio = ~np.isnan(o)                         # el valor original está dentro de arr
        debajo -= (propio & (o < v))
        iguales -= (propio & (o == v))
        n = len(arr) + (~propio).astype(float)        # si no estaba, el proceso entra al conjunto
        r_med = debajo + (iguales + 2) / 2            # rango medio del bloque de empates, con el propio
        r_min, r_max = debajo + 1, debajo + iguales + 1
        f = regla["formula"]
        if f == "mitad_de_salto":
            p = (r_med - 0.5) / n
        elif f == "rango_medio/n":
            p = r_med / n
        elif f.startswith("acumulada"):
            p = r_max / n
        else:
            p = (r_min - 1) / (n - 1)
        salida[m] = p
    return salida

# control: con el valor ORIGINAL, el recálculo debe devolver exactamente el percentil de la base
say("Control del recálculo (valor original -> percentil de la base, % idéntico en 2.000 procesos al azar):")
rng_c = np.random.default_rng(SEMILLA)
for pct, regla in REGLAS.items():
    con_dato = df.index[valores_orientados(df, regla).notna() & num(df[pct]).notna()]
    if len(con_dato) == 0:
        continue
    filas = pd.Index(rng_c.choice(con_dato, size=min(2000, len(con_dato)), replace=False))
    p = recalcular(pct, filas, num(df.loc[filas, regla["fuente"]]).to_numpy())
    igual = 100 * np.mean(np.abs(p - num(df.loc[filas, pct]).to_numpy()) < 1e-6)
    say(f"  {pct:<26} {igual:6.2f} %")
    if igual < 99:
        raise SystemExit(f"El recálculo de {pct} no reproduce la base: pégame esta salida.")

# precio: grupo de referencia, mediana y escala de cada proceso (regla de la celda R)
PRECIO = REPRO["precio"]
t = DF_TODO.copy()
pu = num(t["precio_unit"])
t["lp"] = np.log(pu.where(pu > 0))
t["z"] = num(t["ind_desv_precio_unit_z"])
c5 = t["cpc5"].astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True)
t["cpc5_s"] = c5.where(c5 != "", np.nan).str.zfill(5)
t["cpc3_s"] = t["cpc5_s"].str[:3]
t["unidad_s"] = t["unidad"].astype(str).str.strip().str.upper() if "unidad" in t.columns else "-"
t["unidad_raw"] = t["unidad"] if "unidad" in t.columns else "-"
t["familia_s"] = t["familia"].astype(str)
t["anio_fecha"] = pd.to_datetime(t["fecha_proceso"], errors="coerce").dt.year
niv = t["ind_precio_nivel"].astype("object")
t["m_ref"], t["mad_ref"] = np.nan, np.nan
for etiqueta, cfg in PRECIO["niveles"].items():
    es = niv.astype(str) == etiqueta
    keys, alcance = cfg["claves"], cfg["alcance"]
    m = t["lp"].notna() & (es if alcance == "mismo_nivel" else True)
    sub = t.loc[m, keys + ["lp"]]
    med = sub.groupby(keys, dropna=False)["lp"].transform("median")
    mad = (sub["lp"] - med).abs().groupby([sub[k] for k in keys], dropna=False).transform("median")
    t.loc[es, "m_ref"] = med.reindex(t.index)[es]
    t.loc[es, "mad_ref"] = mad.reindex(t.index)[es]
ESCALA = PRECIO["c_mad"] * t["mad_ref"]
z_hat = (t["lp"] - t["m_ref"]) / ESCALA
t["precio_ok"] = (z_hat - t["z"]).abs() < 1e-6 * np.maximum(1, t["z"].abs())
REF_PRECIO = pd.DataFrame({"ocid": t["ocid"], "escala": ESCALA, "precio_ok": t["precio_ok"],
                           "m_ref": t["m_ref"]}).set_index("ocid")
T_PRECIO = t        # se conserva para la referencia fina de E3b (celda M7)
df["escala_precio"] = df["ocid"].map(REF_PRECIO["escala"])
df["precio_ok"] = df["ocid"].map(REF_PRECIO["precio_ok"]).fillna(False).astype(bool)
df["m_ref_precio"] = df["ocid"].map(REF_PRECIO["m_ref"])
say(f"\nPrecio reproducido en {100 * df['precio_ok'].mean():.2f} % de los procesos puntuables")

# constantes por grupo para las inyecciones (en la base completa, con la agrupación de cada regla)
def percentil_de_grupo(pct, q):
    regla = REGLAS[pct]
    x = num(DF_TODO[regla["fuente"]])
    k = clave_grupo(DF_TODO, regla["agrupacion"])
    return x.groupby(k).quantile(q)

P5_PLAZO = percentil_de_grupo("pct_plazo_corto", 0.05) if "pct_plazo_corto" in REGLAS else pd.Series(dtype=float)
P99_CAPITAL = percentil_de_grupo("pct_monto_capital", 0.99) if "pct_monto_capital" in REGLAS else pd.Series(dtype=float)
ANTIG_NUEVO = 30.0                                # días: proveedor constituido un mes antes
FACTORES = {"precio_x1,5": 1.5, "precio_x2": 2.0, "precio_x3": 3.0}
COMPETITIVAS = ("subasta", "menor cuantia", "cotizacion", "licitacion", "lista corta", "concurso")

def recetas_del_grupo(g, plan, filas24):
    """Para cada tipo de anomalía: procesos elegibles de 2024 y valores nuevos de X (escala de la base)."""
    usar = plan["usar"]
    sub = df.loc[filas24]
    R = {}
    if "precio_alto" in usar and "pct_precio_alto" in REGLAS:
        z = num(sub["ind_desv_precio_unit_z"])
        for nombre, k in FACTORES.items():
            z_nuevo = np.log(k) / sub["escala_precio"]
            ok = sub["precio_ok"] & z.notna() & (z < z_nuevo) & sub["escala_precio"].gt(0)
            # el indicador se recalcula desde el z nuevo (la fuente de pct_precio_alto es el z)
            filas = sub.index[ok]
            R[nombre] = (filas, {"precio_alto": recalcular("pct_precio_alto", filas, z_nuevo[ok].to_numpy())})
    if sin_tildes(g).startswith(COMPETITIVAS) and "oferente_unico" in usar and "ind_n_oferentes" in sub:
        filas = sub.index[num(sub["ind_n_oferentes"]) >= 2]
        cambios = {"oferente_unico": np.ones(len(filas))}
        if "poca_concurrencia" in usar:
            cambios["poca_concurrencia"] = np.ones(len(filas))
        R["oferente_unico"] = (filas, cambios)
    if "plazo_corto" in usar and "pct_plazo_corto" in REGLAS:
        regla = REGLAS["pct_plazo_corto"]
        p5 = clave_grupo(sub, regla["agrupacion"]).map(P5_PLAZO)
        actual = num(sub[regla["fuente"]])
        ok = actual.notna() & p5.notna() & (actual > p5)
        filas = sub.index[ok]
        R["plazo_p5"] = (filas, {"plazo_corto": recalcular("pct_plazo_corto", filas, p5[ok].to_numpy())})
    if {"proveedor_nuevo", "monto_capital"} <= set(usar) and {"pct_proveedor_nuevo", "pct_monto_capital"} <= set(REGLAS):
        r_an, r_mc = REGLAS["pct_proveedor_nuevo"], REGLAS["pct_monto_capital"]
        an, mc = num(sub[r_an["fuente"]]), num(sub[r_mc["fuente"]])
        p99 = clave_grupo(sub, r_mc["agrupacion"]).map(P99_CAPITAL)
        ok = an.notna() & (an > ANTIG_NUEVO) & mc.notna() & p99.notna() & (mc < p99)
        filas = sub.index[ok]
        cambios = {"proveedor_nuevo": recalcular("pct_proveedor_nuevo", filas, np.full(len(filas), ANTIG_NUEVO)),
                   "monto_capital": recalcular("pct_monto_capital", filas, p99[ok].to_numpy())}
        for v, val in [("debutante", 1.0), ("cuota_entidad", 0.0), ("recurrencia_par", 0.0), ("recompra_30d", 0.0)]:
            if v in usar:
                cambios[v] = np.full(len(filas), val)
        R["nuevo_monto_alto"] = (filas, cambios)
    # combinaciones: intersección de elegibles y unión de cambios
    for nombre, (a, b) in {"combinado_precio_oferente": ("precio_x2", "oferente_unico"),
                           "combinado_precio_plazo": ("precio_x2", "plazo_p5"),
                           "combinado_nuevo_precio": ("nuevo_monto_alto", "precio_x1,5")}.items():
        if a in R and b in R:
            filas = R[a][0].intersection(R[b][0])
            cambios = {}
            for parte in (a, b):
                f_p, c_p = R[parte]
                pos = f_p.get_indexer(filas)
                for v, arr in c_p.items():
                    cambios[v] = np.asarray(arr)[pos]
            R[nombre] = (filas, cambios)
    return {k: v for k, v in R.items() if len(v[0]) > 0}

def preparar_anio(anio, minimo=1000):
    """Grupos anteriores a la reforma con al menos `minimo` procesos en `anio` y sus recetas (como H1 con 2024)."""
    grupos, recetas, filas_t = [], {}, []
    for g, plan in PLAN.items():
        if not g.endswith("| pre"):
            continue
        filas_a = plan["idx"][(df.loc[plan["idx"], "anio_proc"] == anio).to_numpy()]
        if len(filas_a) < minimo:
            continue
        grupos.append(g)
        recetas[g] = recetas_del_grupo(g, plan, filas_a)
        fila = dict(grupo=g, procesos=len(filas_a), inyecciones_por_semilla=int(round(CORTE_TOP * len(filas_a))))
        fila.update({k: len(v[0]) for k, v in recetas[g].items()})
        filas_t.append(fila)
    return grupos, recetas, pd.DataFrame(filas_t)

ANIOS = {}
for anio in [2024] + list(ANIOS_CONFIRMACION):
    gr, rc, tb = preparar_anio(anio)
    if anio in ANIOS_CONFIRMACION and len(gr) < 4:
        reemplazo = ANIOS_CONFIRMACION[anio]
        say(f"\n{anio}: solo {len(gr)} grupos con 1.000 procesos o más; se reemplaza por {reemplazo} (enmienda D, 6).")
        ALERTAS.append(f"{anio} reemplazado por {reemplazo}")
        anio = reemplazo
        gr, rc, tb = preparar_anio(anio)
    ANIOS[anio] = dict(grupos=gr, recetas=rc)
    say(f"\nAño {anio}: {len(gr)} grupos; procesos elegibles para cada tipo de anomalía:")
    say(tb.fillna(0).set_index("grupo").T.to_string() if len(tb) else "  (ninguno)")
AN_CONF = [a for a in ANIOS if a != 2024]
say(f"\nAños de confirmación usados: {AN_CONF}")
say(f"[M6: {time.time() - t_celda:.0f} s]")

In [ ]:
# =====================================================================================
# CUADERNO 20 (MOTOR) · CELDA M7. E3b: referencia de precio más fina (enmienda D, 5.5)
#   Celda fina: CPC5 x unidad dentro de la misma familia y año (las claves del nivel "cpc5 x unidad"),
#   dividida en terciles de cantidad del ítem. Si la celda fina tiene menos de 30 procesos con precio
#   (o su MAD es cero), el proceso conserva su referencia vigente. Con el z fino se recalcula
#   pct_precio_alto con la misma regla de la base (mitad de salto dentro de familia x período).
#   Una inyección es el mismo precio en las dos versiones: k veces la mediana de la referencia vigente.
# =====================================================================================
empezar("M7")
t_celda = time.time()
if "cantidad" not in T_PRECIO.columns:
    raise SystemExit("La base no trae la columna 'cantidad': E3b no se puede ejecutar. Para y avisa.")
tp = T_PRECIO
cant = num(tp["cantidad"])
claves_celda = PRECIO["niveles"]["cpc5 x unidad"]["claves"]          # cpc5_s, unidad_s, familia_s, anio
apta = tp["lp"].notna() & cant.gt(0) & tp["cpc5_s"].notna()
sub = tp.loc[apta, claves_celda + ["lp"]].copy()
sub["cant"] = cant[apta]
rp = sub.groupby(claves_celda, dropna=False)["cant"].rank(method="average", pct=True)
sub["tercil"] = np.clip(np.ceil(3 * rp), 1, 3).astype(int)
k_f = claves_celda + ["tercil"]
sub["n_f"] = sub.groupby(k_f, dropna=False)["lp"].transform("size")
sub["m_f"] = sub.groupby(k_f, dropna=False)["lp"].transform("median")
sub["mad_f"] = (sub["lp"] - sub["m_f"]).abs().groupby([sub[k] for k in k_f], dropna=False).transform("median")
usa = (sub["n_f"] >= MIN_CELDA_FINA) & sub["mad_f"].gt(0)
tp["usa_fino"] = False
tp.loc[sub.index[usa], "usa_fino"] = True
tp["m_fino"] = tp["m_ref"]
tp["escala_fina"] = PRECIO["c_mad"] * tp["mad_ref"]
tp.loc[sub.index[usa], "m_fino"] = sub.loc[usa, "m_f"]
tp.loc[sub.index[usa], "escala_fina"] = PRECIO["c_mad"] * sub.loc[usa, "mad_f"]
tp["z_fino"] = tp["z"]
zf = (tp["lp"] - tp["m_fino"]) / tp["escala_fina"]
tp.loc[tp["usa_fino"], "z_fino"] = zf[tp["usa_fino"]]

REGLA_P = REGLAS["pct_precio_alto"]
def pct_regla_base(valores, tabla, regla):
    """Percentil de la base para una fuente nueva: misma agrupación, orientación y fórmula (mitad de salto)."""
    if regla["formula"] != "mitad_de_salto" or regla["subconjunto"] != "todos":
        raise SystemExit("La regla de pct_precio_alto no es la esperada: para y avisa.")
    v = regla["signo"] * pd.Series(np.asarray(valores, dtype=float), index=tabla.index)
    k = clave_grupo(tabla, regla["agrupacion"])
    r = v.groupby(k).rank(method="average")
    n = v.notna().groupby(k).transform("sum")
    return (r - 0.5) / n

# control: la misma función con el z vigente debe devolver el pct_precio_alto de la base
p_ctrl = pct_regla_base(tp["z"], tp, REGLA_P)
base_p = num(tp["pct_precio_alto"])
ok_ctrl = base_p.notna()
igual_ctrl = 100 * np.mean(np.abs(p_ctrl[ok_ctrl] - base_p[ok_ctrl]) < 1e-9)
say(f"Control: el percentil recalculado con el z vigente coincide con la base en {igual_ctrl:.3f} % de los procesos")
if igual_ctrl < 99.9:
    raise SystemExit("El percentil del precio no se reproduce: para y avisa.")
tp["pct_precio_fino"] = pct_regla_base(tp["z_fino"], tp, REGLA_P)

# distribución fina por grupo de la regla (base completa), para recalcular los inyectados
k_regla = clave_grupo(tp, REGLA_P["agrupacion"])
ORDENADOS_FINO = {g: np.sort(s_.dropna().to_numpy()) for g, s_ in (REGLA_P["signo"] * tp["z_fino"]).groupby(k_regla)}

def recalcular_con(regla, ordenados, filas, nuevos, originales):
    """Como recalcular() de la celda M6, pero con otra distribución y otros valores originales."""
    sub_ = df.loc[filas]
    claves = clave_grupo(sub_, regla["agrupacion"]).to_numpy()
    orig = regla["signo"] * np.asarray(originales, dtype=float)
    nuevo = regla["signo"] * np.asarray(nuevos, dtype=float)
    salida = np.full(len(filas), np.nan)
    for g in np.unique(claves):
        m = claves == g
        arr = ordenados.get(g, np.array([]))
        v, o = nuevo[m], orig[m]
        debajo = np.searchsorted(arr, v, side="left").astype(float)
        iguales = np.searchsorted(arr, v, side="right") - debajo
        propio = ~np.isnan(o)
        debajo -= (propio & (o < v))
        iguales -= (propio & (o == v))
        n = len(arr) + (~propio).astype(float)
        salida[m] = (debajo + (iguales + 2) / 2 - 0.5) / n
    return salida

# control: recalcular_con con la distribución vigente y el valor original devuelve el percentil de la base
filas_c = df.index[df["precio_ok"] & num(df["ind_desv_precio_unit_z"]).notna()][:2000]
p_rc = recalcular_con(REGLA_P, ORDENADOS["pct_precio_alto"], filas_c,
                      num(df.loc[filas_c, "ind_desv_precio_unit_z"]), num(df.loc[filas_c, "ind_desv_precio_unit_z"]))
igual_rc = 100 * np.mean(np.abs(p_rc - num(df.loc[filas_c, "pct_precio_alto"]).to_numpy()) < 1e-6)
say(f"Control de recalcular_con: {igual_rc:.2f} % idéntico")
if igual_rc < 99:
    raise SystemExit("recalcular_con no reproduce la base: para y avisa.")

fino = tp.set_index("ocid")[["usa_fino", "m_fino", "escala_fina", "z_fino", "pct_precio_fino"]]
for c in fino.columns:
    df[c] = df["ocid"].map(fino[c])
X_FINO = X.copy()
if "precio_alto" in X_FINO.columns:
    X_FINO["precio_alto"] = a_float(df["pct_precio_fino"]).where(X["precio_alto"].notna())

# descriptivo: cuánto cambia la referencia en los grupos de H1 (desde 2016)
filas_d = []
for g in ANIOS[2024]["grupos"]:
    m = (df["grupo"] == g) & df["escala_precio"].gt(0)
    filas_d.append(dict(grupo=g, procesos_con_precio=int(m.sum()),
                        pct_con_referencia_fina=round(100 * df.loc[m, "usa_fino"].fillna(False).mean(), 1),
                        escala_vigente_mediana=round(df.loc[m, "escala_precio"].median(), 3),
                        escala_fina_mediana=round(df.loc[m, "escala_fina"].median(), 3),
                        z_x3_vigente=round(np.log(3) / df.loc[m, "escala_precio"].median(), 2),
                        z_x3_fino=round(np.log(3) / df.loc[m, "escala_fina"].median(), 2)))
TABLA_FINO = pd.DataFrame(filas_d)
say("\nReferencia fina en los grupos de H1 (procesos puntuables con precio):")
say(TABLA_FINO.to_string(index=False))
say(f"[M7: {time.time() - t_celda:.0f} s]")

In [ ]:
# =====================================================================================
# CUADERNO 20 (MOTOR) · CELDA M8. Motor de inyección y control: repetir la semilla 1 de H1
#   El motor es el bucle de la celda 7 del cuaderno 17, con el año, los tipos de anomalía y las
#   versiones de las variables ("brazos") como parámetros. Antes de cualquier corrida larga se repite
#   la semilla 1 de H1 (2024, todos los tipos) y se compara con h1_inyecciones.csv proceso por proceso.
#   Tarda lo que una semilla de H1 (unos 17 minutos). No sigas si dice que no se reproduce.
# =====================================================================================
empezar("M8")
METODOS = ["ensamble", "base", "if", "lof", "ae"]
CORTES = [0.01, 0.02, 0.05]
TIPOS_PRECIO = ["precio_x1,5", "precio_x2", "precio_x3"]

def cambios_vigentes(recetas_g):
    def f(g, tipo, elegidos, pos):
        return {v: np.asarray(arr)[pos] for v, arr in recetas_g[tipo][1].items()}
    return f

def cambios_finos(recetas_g):
    """Precio: el mismo precio inyectado (k veces la mediana vigente), medido contra la referencia fina."""
    def f(g, tipo, elegidos, pos):
        if tipo not in FACTORES:
            return {v: np.asarray(arr)[pos] for v, arr in recetas_g[tipo][1].items()}
        lp_nuevo = df.loc[elegidos, "m_ref_precio"].to_numpy(float) + np.log(FACTORES[tipo])
        z_nuevo = (lp_nuevo - df.loc[elegidos, "m_fino"].to_numpy(float)) / df.loc[elegidos, "escala_fina"].to_numpy(float)
        return {"precio_alto": recalcular_con(REGLA_P, ORDENADOS_FINO, pd.Index(elegidos), z_nuevo,
                                              df.loc[elegidos, "z_fino"].to_numpy(float))}
    return f

def una_semilla(anio, s, g, gi, tipos_sel, brazos):
    """Inyecta en un grupo y una semilla (misma secuencia aleatoria que H1) y puntúa cada brazo."""
    plan = PLAN[g]
    recetas_g = ANIOS[anio]["recetas"][g]
    filas_a = plan["idx"][(df.loc[plan["idx"], "anio_proc"] == anio).to_numpy()]
    rng = np.random.default_rng([s, gi])
    tipos = [t_ for t_ in recetas_g if t_ in tipos_sel]
    if not tipos:
        return []
    n_iny = max(1, int(round(CORTE_TOP * len(filas_a))))
    orden = [tipos[i] for i in rng.permutation(len(tipos))]
    cuantos = pd.Series([orden[i % len(orden)] for i in range(n_iny)]).value_counts()
    usados, marca, elegidos_por_tipo = set(), pd.Series("", index=filas_a), []
    for tipo in orden:
        if tipo not in cuantos.index:
            continue
        filas_ok, _ = recetas_g[tipo]
        libres = np.array([i for i in filas_ok if i not in usados])
        if len(libres) == 0:
            continue
        elegidos = rng.choice(libres, size=min(int(cuantos[tipo]), len(libres)), replace=False)
        elegidos_por_tipo.append((tipo, elegidos, filas_ok.get_indexer(elegidos)))
        usados.update(elegidos.tolist())
        marca.loc[elegidos] = tipo
    iny = marca[marca != ""]
    salida = []
    for brazo, (X_brazo, fabrica) in brazos.items():
        cambia = fabrica(recetas_g)
        Xg = X_brazo.loc[plan["idx"], plan["usar"]].copy()
        for tipo, elegidos, pos in elegidos_por_tipo:
            for v, arr in cambia(g, tipo, elegidos, pos).items():
                if v in Xg.columns:
                    Xg.loc[elegidos, v] = arr
        r, _, _ = puntuar_grupo(plan, Xg, SEMILLA + s)
        ra = r.loc[filas_a]
        for metodo in METODOS:
            pctl = ra[metodo].rank(pct=True)
            tops = {c: pd.Series(top_k(ra[metodo], c, SEMILLA + s), index=ra.index) for c in CORTES}
            for i, tipo in iny.items():
                salida.append(dict(anio=anio, brazo=brazo, semilla=s, grupo=g, proceso=int(i), tipo=tipo,
                                   metodo=metodo, percentil=float(pctl.loc[i]),
                                   **{f"en_top_{int(c * 100)}": bool(tops[c].loc[i]) for c in CORTES}))
    return salida

def correr(nombre, plan_anios, tipos_sel, brazos, carpeta=None):
    """Corre (o reanuda) un análisis. plan_anios = {año: [semillas]}. Guarda el avance tras cada semilla."""
    if carpeta is None:
        carpeta = os.path.join(SALIDA, datetime.now().strftime("%Y%m%d_%H%M") + f"_expD_{nombre}")
        os.makedirs(carpeta, exist_ok=False)
    archivo = os.path.join(carpeta, f"{nombre}_inyecciones.csv")
    previos = pd.read_csv(archivo) if os.path.exists(archivo) else pd.DataFrame()
    hechos = set(map(tuple, previos[["anio", "semilla"]].drop_duplicates().to_numpy())) if len(previos) else set()
    registros = previos.to_dict("records")
    pendientes = [(a, s) for a, ss in plan_anios.items() for s in ss if (a, s) not in hechos]
    say(f"{nombre}: {len(pendientes)} semillas por correr ({len(hechos)} ya hechas) en {carpeta}")
    t0 = time.time()
    for j, (a, s) in enumerate(pendientes, start=1):
        ts = time.time()
        grupos = ANIOS[a]["grupos"]
        for gi, g in enumerate(grupos):
            registros.extend(una_semilla(a, s, g, gi, tipos_sel, brazos))
        pd.DataFrame(registros).to_csv(archivo, index=False)
        resta = (time.time() - t0) / j * (len(pendientes) - j) / 60
        say(f"  {nombre} {a} semilla {s:>2}: {time.time() - ts:.0f} s | faltan unos {resta:.0f} min "
            f"({datetime.now():%d-%b %H:%M})")
    return carpeta, pd.DataFrame(registros)

# ---- control: semilla 1 de H1 ----
t_celda = time.time()
TODOS_LOS_TIPOS = {t_ for a_ in ANIOS for g in ANIOS[a_]["grupos"] for t_ in ANIOS[a_]["recetas"][g]}
CARPETA_CONTROL, CTRL = correr("control", {2024: [1]}, TODOS_LOS_TIPOS, {"normal_sup": (X, cambios_vigentes)})
H1_ORIG = pd.read_csv(os.path.join(H1_DIR, "h1_inyecciones.csv"))
h1s = H1_ORIG[H1_ORIG["semilla"] == 1]
claves_c = ["semilla", "grupo", "proceso", "tipo", "metodo"]
cmp_ = h1s.merge(CTRL, on=claves_c, how="outer", suffixes=("_h1", "_motor"), indicator=True)
solo = cmp_["_merge"].value_counts().to_dict()
ambos = cmp_[cmp_["_merge"] == "both"]
igual = {c: round(100 * float((ambos[f"{c}_h1"] == ambos[f"{c}_motor"]).mean()), 3) for c in ["en_top_1", "en_top_2", "en_top_5"]}
dif_p = float((ambos["percentil_h1"] - ambos["percentil_motor"]).abs().max()) if len(ambos) else float("nan")
say(f"\nControl contra H1 (semilla 1): filas en ambos {solo.get('both', 0):,}, solo H1 {solo.get('left_only', 0)}, "
    f"solo motor {solo.get('right_only', 0)}")
say(f"Coincidencia de aciertos: {igual} | diferencia máxima de percentil {dif_p:.2e}")
REPRODUCE = solo.get("left_only", 0) == 0 and solo.get("right_only", 0) == 0 and min(igual.values()) >= 99.5
json.dump(dict(corrida=os.path.basename(CARPETA_CONTROL), huella_codigo=HUELLA_CODIGO, enmienda_d=HUELLA_D,
               filas=solo, coincidencia=igual, dif_percentil_max=dif_p, reproduce=REPRODUCE, versiones=VERSIONES,
               minutos=round((time.time() - t_celda) / 60, 1)),
          open(os.path.join(CARPETA_CONTROL, "registro_control.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
say("REPRODUCE H1: sí. Se puede seguir con E0 y E3b." if REPRODUCE else
    "NO REPRODUCE H1 exactamente: para y avisa antes de correr nada más.")
say(f"[M8: {(time.time() - t_celda) / 60:.1f} min]")

In [ ]:
# =====================================================================================
# CUADERNO 20 (MOTOR) · CELDA M9. E0: comparación de métodos en años sin uso (enmienda D, 5.2)
#   H1 repetido con normal_sup y los cinco métodos, inyectando en los años de confirmación
#   (2022 y 2019, o sus reemplazos) con las semillas 21 a 40. 2024 sale de H1 sin volver a correr.
#   Larga (unas 11 horas). Guarda el avance tras cada semilla: si se corta, pon en CARPETA_E0 la
#   carpeta que ya existe (por ejemplo r"C:\SERCOP_FINAL\modelos\20261122_1530_expD_E0") y vuelve a correr.
# =====================================================================================
empezar("M9")
if not REPRODUCE:
    raise SystemExit("El control de la celda M8 no reprodujo H1: no se corre E0.")
CARPETA_E0 = None
CARPETA_E0, E0R = correr("E0", {a: SEMILLAS_CONFIRMACION for a in AN_CONF}, TODOS_LOS_TIPOS,
                         {"normal_sup": (X, cambios_vigentes)}, carpeta=CARPETA_E0)
say(f"E0 terminado: {len(E0R):,} filas en {CARPETA_E0}")

In [ ]:
# =====================================================================================
# CUADERNO 20 (MOTOR) · CELDA M10. E3b: inyecciones de precio con la referencia fina (enmienda D, 5.5)
#   Solo los tres tipos de precio (1,5, 2 y 3 veces la mediana). En cada semilla, grupo y año se eligen
#   los mismos procesos y el mismo precio para las dos versiones: "vigente" (la referencia de la base)
#   y "fino" (CPC5 x unidad x terciles de cantidad). Se reajustan los modelos en cada versión.
#   2024 con las semillas 1 a 20 y los años de confirmación con las semillas 21 a 40.
#   La más larga (unas 35 horas). Para reanudar, pon en CARPETA_E3B la carpeta que ya existe.
# =====================================================================================
empezar("M10")
if not REPRODUCE:
    raise SystemExit("El control de la celda M8 no reprodujo H1: no se corre E3b.")
CARPETA_E3B = None
plan_e3b = {2024: SEMILLAS_EXPLORACION, **{a: SEMILLAS_CONFIRMACION for a in AN_CONF}}
CARPETA_E3B, E3BR = correr("E3b", plan_e3b, set(TIPOS_PRECIO),
                           {"vigente": (X, cambios_vigentes), "fino": (X_FINO, cambios_finos)}, carpeta=CARPETA_E3B)
say(f"E3b terminado: {len(E3BR):,} filas en {CARPETA_E3B}")

In [ ]:
# =====================================================================================
# CUADERNO 20 (MOTOR) · CELDA M11. Contrastes, regla de confirmación y registros (enmienda D, 5.1 y 6)
#   Cada contraste es la diferencia media de aciertos en el 1 % superior entre dos métodos (o dos
#   versiones) sobre las mismas inyecciones, con IC 95 % por remuestreo de semillas (2.000 réplicas),
#   como en H1. Confirmado si el IC de 2022 queda del lado favorable de cero y 2019 tiene el mismo signo.
#   Se puede correr cuando termine E0, cuando termine E3b o al final; usa lo que exista.
# =====================================================================================
empezar("M11")
def contraste(R, col, a, b, valor="en_top_1"):
    par = R.pivot_table(index=["semilla", "grupo", "proceso", "tipo"], columns=col, values=valor).astype(float)
    if a not in par or b not in par:
        return None
    par = par.dropna(subset=[a, b])
    d = par[a] - par[b]
    ps = d.groupby(level="semilla").agg(["sum", "count"])
    rng_b = np.random.default_rng(SEMILLA)
    sem = ps.index.to_numpy()
    boot = np.empty(B_REMUESTREO)
    for i in range(B_REMUESTREO):
        sel = ps.loc[rng_b.choice(sem, size=len(sem), replace=True)]
        boot[i] = sel["sum"].sum() / sel["count"].sum()
    ic = np.percentile(boot, [2.5, 97.5])
    return dict(inyecciones=int(len(par)), semillas=int(len(sem)), rec_a=round(float(par[a].mean()), 4),
                rec_b=round(float(par[b].mean()), 4), D=round(float(d.mean()), 4),
                ic95_inf=round(float(ic[0]), 4), ic95_sup=round(float(ic[1]), 4))

def confirmar(filas, principal, favorable=+1):
    """Regla de la enmienda D: IC de 2022 del lado favorable y 2019 con el mismo signo."""
    f = {r["anio"]: r for r in filas if r["contraste"] == principal}
    a1, a2 = AN_CONF[0], AN_CONF[1] if len(AN_CONF) > 1 else None
    if a1 not in f or a2 not in f:
        return "incompleto"
    ok1 = (f[a1]["ic95_inf"] > 0) if favorable > 0 else (f[a1]["ic95_sup"] < 0)
    ok2 = np.sign(f[a2]["D"]) == favorable
    return "CONFIRMADO" if (ok1 and ok2) else "no confirmado"

REGISTROS_EXP = {}
# ---- E0 ----
if "E0R" in globals() and len(E0R):
    h1 = pd.read_csv(os.path.join(H1_DIR, "h1_inyecciones.csv")).assign(anio=2024)
    filas = []
    for anio, R in [(2024, h1)] + [(a, E0R[E0R["anio"] == a]) for a in AN_CONF]:
        for nombre, a, b in [("C0a ensamble - indice", "ensamble", "base"),
                             ("C0b IF - indice", "if", "base"), ("C0c IF - ensamble", "if", "ensamble")]:
            r = contraste(R, "metodo", a, b)
            if r:
                filas.append(dict(contraste=nombre, anio=anio, **r))
    T0 = pd.DataFrame(filas)
    say("E0. Contrastes por año (2024 = H1; los demás, años de confirmación):")
    say(T0.to_string(index=False))
    dec = {c: confirmar(filas, c) for c in ["C0b IF - indice", "C0c IF - ensamble"]}
    afirma = "CONFIRMADA" if all(v == "CONFIRMADO" for v in dec.values()) else "no confirmada"
    say(f"Regla de confirmación: {dec} -> 'Isolation Forest recupera más que el ensamble y que el índice' {afirma}")
    rec = E0R.groupby(["anio", "metodo"])["en_top_1"].mean().unstack().round(4)
    say("\nRecuperación al 1 % por año y método (descriptivo):")
    say(rec.to_string())
    T0.to_csv(os.path.join(CARPETA_E0, "E0_contrastes.csv"), index=False)
    REGISTROS_EXP["E0"] = dict(carpeta=CARPETA_E0, contrastes=filas, confirmacion=dec, afirmacion=afirma)

# ---- E3b ----
if "E3BR" in globals() and len(E3BR):
    filas = []
    for anio in sorted(E3BR["anio"].unique(), key=lambda a: (a != 2024, -a)):
        R = E3BR[E3BR["anio"] == anio]
        for metodo in ["ensamble", "base", "if"]:
            r = contraste(R[R["metodo"] == metodo], "brazo", "fino", "vigente")
            if r:
                filas.append(dict(contraste=f"C3 fino - vigente ({metodo})", anio=int(anio), **r))
    T3 = pd.DataFrame(filas)
    say("\nE3b. Recuperación de las inyecciones de precio con la referencia fina menos con la vigente:")
    say(T3.to_string(index=False))
    dec3 = confirmar(filas, "C3 fino - vigente (ensamble)")
    say(f"Regla de confirmación (C3, ensamble): {dec3}")
    rt = E3BR[E3BR["metodo"] == "ensamble"].pivot_table(index=["anio", "tipo"], columns="brazo", values="en_top_1").round(4)
    say("\nRecuperación al 1 % del ensamble por año, tipo y versión (descriptivo):")
    say(rt.to_string())
    T3.to_csv(os.path.join(CARPETA_E3B, "E3b_contrastes.csv"), index=False)
    REGISTROS_EXP["E3b"] = dict(carpeta=CARPETA_E3B, contrastes=filas, confirmacion=dec3)

for k, v in REGISTROS_EXP.items():
    reg = dict(analisis=k, estatus="exploratorio, enmienda D", enmienda_d=HUELLA_D, huella_codigo=HUELLA_CODIGO,
               preregistro=FIRMA["huella_documento"], anios_confirmacion=AN_CONF, versiones=VERSIONES,
               fecha=datetime.now().isoformat(timespec="seconds"), **v)
    with open(os.path.join(v["carpeta"], f"registro_{k}.json"), "w", encoding="utf-8") as f:
        json.dump(reg, f, ensure_ascii=False, indent=1, default=float)
    with open(os.path.join(v["carpeta"], "resumen_para_chat.txt"), "w", encoding="utf-8") as f:
        for celda, lineas in SALIDA_TXT.items():
            f.write(f"\n##### {celda} #####\n" + "\n".join(lineas) + "\n")
    say(f"Registro escrito en {v['carpeta']}")
say("Pega esta salida en el chat (no trae RUC ni procesos uno por uno).")

In [ ]:
# =====================================================================================
# CUADERNO 20 (MOTOR) · CELDA M12. E1: representación que conserva la magnitud (enmienda D, 5.3)
#   magnitud_sup cambia solo cómo ven los modelos las cinco variables continuas; las otras seis
#   siguen en normal_sup y el índice simple no cambia.
#     precio_alto     : el z de precio tal cual
#     monto_capital   : ln(monto / capital)
#     proveedor_nuevo : -ln(antigüedad en días + 1)
#     plazo_corto     : -ln(plazo de ofertas en días + 1)
#     firma_rapida    : -ln(días entre adjudicación y firma + 1)
#   En las cuatro últimas se resta la mediana del grupo y se divide entre 1,4826 x MAD del grupo
#   (distribución original de la base completa, misma agrupación familia x período que los percentiles).
#   Las cinco se cortan entre 0 y 4. Si el argumento del logaritmo no es positivo, el dato falta.
#   Si la MAD de una variable en un grupo es cero, esa variable sigue en normal_sup en ese grupo.
#   Las inyecciones son las mismas de H1 (2024) y de E0 (2022, 2019): misma secuencia aleatoria.
#   Los valores inyectados se transforman con la misma regla.
# =====================================================================================
empezar("M12")
t_celda = time.time()
MAG = {"precio_alto": ("ind_desv_precio_unit_z", None, None),
       "monto_capital": ("ind_monto_sobre_capital", +1, 0.0),
       "proveedor_nuevo": ("ind_antiguedad_dias", -1, 1.0),
       "plazo_corto": ("ind_plazo_ofertas_dias", -1, 1.0),
       "firma_rapida": ("ind_plazo_adj_firma_dias", -1, 1.0)}
CLAVE_MAG = clave_grupo(DF_TODO, "familia|periodo")
ESTAD_MAG = {}                     # (variable) -> DataFrame por grupo con mediana y escala del logaritmo
for v, (fuente, signo, off) in MAG.items():
    if signo is None:
        continue
    x = num(DF_TODO[fuente]) + off
    L = np.log(x.where(x > 0))
    med = L.groupby(CLAVE_MAG).median()
    mad = (L - CLAVE_MAG.map(med)).abs().groupby(CLAVE_MAG).median()
    ESTAD_MAG[v] = pd.DataFrame({"mediana": med, "escala": PRECIO["c_mad"] * mad})

def a_magnitud(v, valores, claves):
    """Transforma valores de origen de la variable v (en la escala de la base) a magnitud_sup."""
    fuente, signo, off = MAG[v]
    valores = np.asarray(valores, dtype=float)
    if signo is None:
        return np.clip(valores, 0, 4)
    e = ESTAD_MAG[v]
    m = pd.Series(claves).map(e["mediana"]).to_numpy(float)
    s = pd.Series(claves).map(e["escala"]).to_numpy(float)
    arg = valores + off
    with np.errstate(divide="ignore", invalid="ignore"):
        L = np.where(arg > 0, np.log(np.where(arg > 0, arg, 1.0)), np.nan)
        z = signo * (L - m) / s
    return np.clip(z, 0, 4)

CLAVE_DF = clave_grupo(df, "familia|periodo").to_numpy()
X_MAG = pd.DataFrame(index=df.index)
for v, (fuente, _, _) in MAG.items():
    if v in X.columns:
        X_MAG[v] = a_magnitud(v, num(df[fuente]), CLAVE_DF)

# variables con magnitud en cada grupo (MAD > 0) y plan de modelos con la disponibilidad de magnitud_sup
VARS_MAG, PLAN_MAG, filas_m = {}, {}, []
for g in sorted({g for a in ANIOS for g in ANIOS[a]["grupos"]}):
    plan = PLAN[g]
    k_base = clave_grupo(df.loc[plan["idx"][:1]], "familia|periodo").iloc[0]
    vm = []
    for v in MAG:
        if v not in plan["usar"]:
            continue
        if v != "precio_alto":
            esc = ESTAD_MAG[v]["escala"].get(k_base, np.nan)
            if not (esc > 0):
                ALERTAS.append(f"E1: {v} sigue en normal_sup en {g} (MAD cero)")
                continue
        vm.append(v)
    VARS_MAG[g] = vm
    A = X.loc[plan["idx"], plan["usar"]].copy()
    for v in vm:
        A[v] = A[v].where(X_MAG.loc[plan["idx"], v].notna())
    clave, conteo, asign, vars_modelo, grandes = plan_de_modelos(A, plan["usar"])
    PLAN_MAG[g] = dict(idx=plan["idx"], usar=plan["usar"], clave=clave, asign=asign, vars_modelo=vars_modelo, grandes=grandes)
    perdidos = {v: int((X.loc[plan["idx"], v].notna() & X_MAG.loc[plan["idx"], v].isna()).sum()) for v in vm}
    filas_m.append(dict(grupo=g, variables_con_magnitud=", ".join(vm),
                        datos_que_faltan_por_logaritmo=sum(perdidos.values()),
                        **{f"mediana_{v}": round(float(np.nanmedian(X_MAG.loc[plan["idx"], v])), 3) for v in vm}))
TABLA_MAG = pd.DataFrame(filas_m)
say("magnitud_sup por grupo (variables que la usan, datos que faltan porque el logaritmo no existe, medianas):")
say(TABLA_MAG.to_string(index=False))

def puntuar_grupo_mag(plan_m, Xg_pct, Xg_mag, semilla, vm):
    """Como puntuar_grupo(), pero los modelos ven magnitud_sup en las variables vm. El índice simple sale
    de los percentiles, igual que en normal_sup."""
    res = pd.DataFrame(index=Xg_pct.index)
    usar = plan_m["usar"]
    Xm = entrada_modelo(Xg_pct[usar])
    A = Xg_pct[usar].copy()
    for v in vm:
        Xm[v] = Xg_mag[v]
        A[v] = A[v].where(Xg_mag[v].notna())
    for m in ["if", "lof", "ae"]:
        res[m] = np.nan
    res["modelo"] = ""
    modelo_de = plan_m["clave"].loc[Xg_pct.index].map(plan_m["asign"])
    for mk, vs in plan_m["vars_modelo"].items():
        destino = modelo_de.index[modelo_de == mk]
        if len(destino) == 0 or len(vs) == 0:
            continue
        tr_mask = A[vs].notna().all(axis=1)
        if tr_mask.sum() < 30:
            continue
        tope = (REF_MAX, AE_MAX_ENTRENO) if mk in plan_m["grandes"] else (HUERFANO_MAX, HUERFANO_MAX)
        out, _ = ajustar_un_modelo(Xm.loc[tr_mask, vs].to_numpy(dtype=float),
                                   Xm.loc[destino, vs].to_numpy(dtype=float), semilla, *tope)
        for m in out:
            res.loc[destino, m] = out[m]
        res.loc[destino, "modelo"] = mk
    res["ensamble"] = res[["if", "lof", "ae"]].mean(axis=1, skipna=True)
    rz = rareza(Xg_pct[usar])
    orient = [v for v in ORIENTADAS if v in usar]
    res["base"] = rz[orient].mean(axis=1, skipna=True) if orient else np.nan
    return res

PARTES = {"precio_x1,5": [("precio", 1.5)], "precio_x2": [("precio", 2.0)], "precio_x3": [("precio", 3.0)],
          "plazo_p5": [("plazo", None)], "nuevo_monto_alto": [("nuevo", None)], "oferente_unico": [],
          "combinado_precio_oferente": [("precio", 2.0)], "combinado_precio_plazo": [("precio", 2.0), ("plazo", None)],
          "combinado_nuevo_precio": [("nuevo", None), ("precio", 1.5)]}

def cambios_magnitud(g, tipo, elegidos):
    """Valores de origen nuevos de cada inyección, llevados a magnitud_sup (solo variables con magnitud)."""
    sub = df.loc[elegidos]
    k = clave_grupo(sub, "familia|periodo").to_numpy()
    c = {}
    for parte, factor in PARTES[tipo]:
        if parte == "precio":
            c["precio_alto"] = a_magnitud("precio_alto", np.log(factor) / sub["escala_precio"].to_numpy(float), k)
        elif parte == "plazo":
            p5 = clave_grupo(sub, REGLAS["pct_plazo_corto"]["agrupacion"]).map(P5_PLAZO).to_numpy(float)
            c["plazo_corto"] = a_magnitud("plazo_corto", p5, k)
        elif parte == "nuevo":
            p99 = clave_grupo(sub, REGLAS["pct_monto_capital"]["agrupacion"]).map(P99_CAPITAL).to_numpy(float)
            c["proveedor_nuevo"] = a_magnitud("proveedor_nuevo", np.full(len(sub), ANTIG_NUEVO), k)
            c["monto_capital"] = a_magnitud("monto_capital", p99, k)
    return {v: arr for v, arr in c.items() if v in VARS_MAG[g]}

def elegir(anio, s, g, gi, tipos_sel):
    """La misma elección de procesos y tipos que una_semilla() (celda M8)."""
    plan = PLAN[g]
    recetas_g = ANIOS[anio]["recetas"][g]
    filas_a = plan["idx"][(df.loc[plan["idx"], "anio_proc"] == anio).to_numpy()]
    rng = np.random.default_rng([s, gi])
    tipos = [t_ for t_ in recetas_g if t_ in tipos_sel]
    n_iny = max(1, int(round(CORTE_TOP * len(filas_a))))
    orden = [tipos[i] for i in rng.permutation(len(tipos))]
    cuantos = pd.Series([orden[i % len(orden)] for i in range(n_iny)]).value_counts()
    usados, marca, elegidos_por_tipo = set(), pd.Series("", index=filas_a), []
    for tipo in orden:
        if tipo not in cuantos.index:
            continue
        filas_ok, _ = recetas_g[tipo]
        libres = np.array([i for i in filas_ok if i not in usados])
        if len(libres) == 0:
            continue
        elegidos = rng.choice(libres, size=min(int(cuantos[tipo]), len(libres)), replace=False)
        elegidos_por_tipo.append((tipo, elegidos, filas_ok.get_indexer(elegidos)))
        usados.update(elegidos.tolist())
        marca.loc[elegidos] = tipo
    return filas_a, elegidos_por_tipo, marca[marca != ""]

def una_semilla_mag(anio, s, g, gi, tipos_sel):
    plan = PLAN[g]
    recetas_g = ANIOS[anio]["recetas"][g]
    filas_a, elegidos_por_tipo, iny = elegir(anio, s, g, gi, tipos_sel)
    Xp = X.loc[plan["idx"], plan["usar"]].copy()
    Xmg = X_MAG.loc[plan["idx"]].copy()
    for tipo, elegidos, pos in elegidos_por_tipo:
        for v, arr in recetas_g[tipo][1].items():
            if v in Xp.columns:
                Xp.loc[elegidos, v] = np.asarray(arr)[pos]
        for v, arr in cambios_magnitud(g, tipo, elegidos).items():
            Xmg.loc[elegidos, v] = arr
    r = puntuar_grupo_mag(PLAN_MAG[g], Xp, Xmg, SEMILLA + s, VARS_MAG[g])
    ra = r.loc[filas_a]
    salida = []
    for metodo in METODOS:
        pctl = ra[metodo].rank(pct=True)
        tops = {c: pd.Series(top_k(ra[metodo], c, SEMILLA + s), index=ra.index) for c in CORTES}
        for i, tipo in iny.items():
            salida.append(dict(anio=anio, brazo="magnitud_sup", semilla=s, grupo=g, proceso=int(i), tipo=tipo,
                               metodo=metodo, percentil=float(pctl.loc[i]),
                               **{f"en_top_{int(c * 100)}": bool(tops[c].loc[i]) for c in CORTES}))
    return salida

# control: la elección de elegir() es la misma que la de una_semilla() (semilla 1, primer grupo de 2024)
g0 = ANIOS[2024]["grupos"][0]
_, _, iny0 = elegir(2024, 1, g0, 0, TODOS_LOS_TIPOS)
h1_1 = pd.read_csv(os.path.join(H1_DIR, "h1_inyecciones.csv"))
h1_1 = h1_1[(h1_1["semilla"] == 1) & (h1_1["grupo"] == g0) & (h1_1["metodo"] == "ensamble")]
igual_el = set(zip(iny0.index.astype(int), iny0.values)) == set(zip(h1_1["proceso"], h1_1["tipo"]))
say(f"\nControl: elegir() reproduce los procesos y tipos de H1 (semilla 1, {g0}): {'sí' if igual_el else 'NO'}")
if not igual_el:
    raise SystemExit("La elección de inyecciones no coincide con H1: para y avisa.")
say(f"[M12: {time.time() - t_celda:.0f} s]")

In [ ]:
# =====================================================================================
# CUADERNO 20 (MOTOR) · CELDA M13. E1: correr magnitud_sup sobre las mismas inyecciones de H1 y de E0
#   2024 con las semillas 1 a 20 (inyecciones de H1) y 2022 y 2019 con las semillas 21 a 40 (las de E0).
#   La versión normal_sup ya existe: sale de h1_inyecciones.csv y de E0_inyecciones.csv.
#   Unas 2 horas. Guarda tras cada semilla; para reanudar, pon en CARPETA_E1 la carpeta que ya existe.
# =====================================================================================
empezar("M13")
if not REPRODUCE:
    raise SystemExit("El control de la celda M8 no reprodujo H1: no se corre E1.")
CARPETA_E1 = None

def correr_mag(nombre, plan_anios, tipos_sel, carpeta=None):
    if carpeta is None:
        carpeta = os.path.join(SALIDA, datetime.now().strftime("%Y%m%d_%H%M") + f"_expD_{nombre}")
        os.makedirs(carpeta, exist_ok=False)
    archivo = os.path.join(carpeta, f"{nombre}_inyecciones.csv")
    previos = pd.read_csv(archivo) if os.path.exists(archivo) else pd.DataFrame()
    hechos = set(map(tuple, previos[["anio", "semilla"]].drop_duplicates().to_numpy())) if len(previos) else set()
    registros = previos.to_dict("records")
    pendientes = [(a, s) for a, ss in plan_anios.items() for s in ss if (a, s) not in hechos]
    say(f"{nombre}: {len(pendientes)} semillas por correr ({len(hechos)} ya hechas) en {carpeta}")
    t0 = time.time()
    for j, (a, s) in enumerate(pendientes, start=1):
        ts = time.time()
        for gi, g in enumerate(ANIOS[a]["grupos"]):
            registros.extend(una_semilla_mag(a, s, g, gi, tipos_sel))
        pd.DataFrame(registros).to_csv(archivo, index=False)
        resta = (time.time() - t0) / j * (len(pendientes) - j) / 60
        say(f"  {nombre} {a} semilla {s:>2}: {time.time() - ts:.0f} s | faltan unos {resta:.0f} min ({datetime.now():%d-%b %H:%M})")
    return carpeta, pd.DataFrame(registros)

plan_e1 = {2024: SEMILLAS_EXPLORACION, **{a: SEMILLAS_CONFIRMACION for a in AN_CONF}}
CARPETA_E1, E1R = correr_mag("E1", plan_e1, TODOS_LOS_TIPOS, carpeta=CARPETA_E1)
say(f"E1 terminado: {len(E1R):,} filas en {CARPETA_E1}")

In [ ]:
# =====================================================================================
# CUADERNO 20 (MOTOR) · CELDA M14. E1: contrastes y regla de confirmación (enmienda D, 5.3 y 6)
#   C1 (principal): ensamble con magnitud_sup menos ensamble con normal_sup, sobre las mismas inyecciones.
#   Sin regla de confirmación: los mismos contrastes para Isolation Forest, LOF y autoencoder.
#   Comprueba antes que las inyecciones de las dos versiones son exactamente las mismas y que el índice
#   simple no cambia.
# =====================================================================================
empezar("M14")
# funciones de contraste (las mismas de la celda M11, repetidas para no depender de haberla corrido)
def contraste(R, col, a, b, valor="en_top_1"):
    par = R.pivot_table(index=["semilla", "grupo", "proceso", "tipo"], columns=col, values=valor).astype(float)
    if a not in par or b not in par:
        return None
    par = par.dropna(subset=[a, b])
    d = par[a] - par[b]
    ps = d.groupby(level="semilla").agg(["sum", "count"])
    rng_b = np.random.default_rng(SEMILLA)
    sem = ps.index.to_numpy()
    boot = np.empty(B_REMUESTREO)
    for i in range(B_REMUESTREO):
        sel = ps.loc[rng_b.choice(sem, size=len(sem), replace=True)]
        boot[i] = sel["sum"].sum() / sel["count"].sum()
    ic = np.percentile(boot, [2.5, 97.5])
    return dict(inyecciones=int(len(par)), semillas=int(len(sem)), rec_a=round(float(par[a].mean()), 4),
                rec_b=round(float(par[b].mean()), 4), D=round(float(d.mean()), 4),
                ic95_inf=round(float(ic[0]), 4), ic95_sup=round(float(ic[1]), 4))

def confirmar(filas, principal, favorable=+1):
    """Regla de la enmienda D: IC de 2022 del lado favorable y 2019 con el mismo signo."""
    f = {r["anio"]: r for r in filas if r["contraste"] == principal}
    a1, a2 = AN_CONF[0], AN_CONF[1] if len(AN_CONF) > 1 else None
    if a1 not in f or a2 not in f:
        return "incompleto"
    ok1 = (f[a1]["ic95_inf"] > 0) if favorable > 0 else (f[a1]["ic95_sup"] < 0)
    ok2 = np.sign(f[a2]["D"]) == favorable
    return "CONFIRMADO" if (ok1 and ok2) else "no confirmado"

CARPETA_E0_REAL = CARPETA_E0 if "CARPETA_E0" in globals() and CARPETA_E0 else None
if CARPETA_E0_REAL is None:
    cands = sorted(d for d in os.listdir(SALIDA) if d.endswith("_expD_E0"))
    CARPETA_E0_REAL = os.path.join(SALIDA, cands[-1])
NS = pd.concat([pd.read_csv(os.path.join(H1_DIR, "h1_inyecciones.csv")).assign(anio=2024),
                pd.read_csv(os.path.join(CARPETA_E0_REAL, "E0_inyecciones.csv"))], ignore_index=True)
NS["brazo"] = "normal_sup"
MG = E1R.copy()
claves = ["anio", "semilla", "grupo", "proceso", "tipo", "metodo"]
NS = NS[NS.set_index(["anio", "semilla"]).index.isin(MG.set_index(["anio", "semilla"]).index.unique())]
chk = NS[claves].merge(MG[claves], on=claves, how="outer", indicator=True)["_merge"].value_counts().to_dict()
say(f"Inyecciones emparejadas: ambas {chk.get('both', 0):,}, solo normal_sup {chk.get('left_only', 0)}, "
    f"solo magnitud_sup {chk.get('right_only', 0)}")
if chk.get("left_only", 0) or chk.get("right_only", 0):
    raise SystemExit("Las inyecciones de las dos versiones no son las mismas: para y avisa.")
b = NS[NS["metodo"] == "base"].merge(MG[MG["metodo"] == "base"], on=claves, suffixes=("_n", "_m"))
igual_base = 100 * float((b["en_top_1_n"] == b["en_top_1_m"]).mean())
say(f"Índice simple igual en las dos versiones: {igual_base:.2f} % de las inyecciones")
R1 = pd.concat([NS, MG], ignore_index=True)
filas = []
for anio in [2024] + AN_CONF:
    Ra = R1[R1["anio"] == anio]
    for metodo in ["ensamble", "if", "lof", "ae"]:
        r = contraste(Ra[Ra["metodo"] == metodo], "brazo", "magnitud_sup", "normal_sup")
        if r:
            filas.append(dict(contraste=f"C1 magnitud - normal ({metodo})", anio=int(anio), **r))
T1 = pd.DataFrame(filas)
say("\nE1. Recuperación al 1 % con magnitud_sup menos con normal_sup, por año y método:")
say(T1.to_string(index=False))
dec1 = confirmar(filas, "C1 magnitud - normal (ensamble)")
say(f"Regla de confirmación (C1, ensamble): {dec1}")
rt = R1[R1["metodo"].isin(["ensamble", "if", "lof", "ae"])].pivot_table(index=["metodo", "tipo"], columns=["anio", "brazo"],
                                                                       values="en_top_1").round(3)
say("\nRecuperación al 1 % por método, tipo, año y versión (descriptivo):")
say(rt.to_string())
T1.to_csv(os.path.join(CARPETA_E1, "E1_contrastes.csv"), index=False)
TABLA_MAG.to_csv(os.path.join(CARPETA_E1, "E1_magnitud_por_grupo.csv"), index=False)
reg = dict(analisis="E1", estatus="exploratorio, enmienda D 5.3", enmienda_d=HUELLA_D, huella_codigo=HUELLA_CODIGO,
           preregistro=FIRMA["huella_documento"], anios_confirmacion=AN_CONF, versiones=VERSIONES,
           normal_sup_de=[os.path.basename(H1_DIR), os.path.basename(CARPETA_E0_REAL)], emparejadas=chk,
           indice_igual_pct=igual_base, contrastes=filas, confirmacion=dec1, alertas=ALERTAS,
           fecha=datetime.now().isoformat(timespec="seconds"), carpeta=CARPETA_E1)
with open(os.path.join(CARPETA_E1, "registro_E1.json"), "w", encoding="utf-8") as f:
    json.dump(reg, f, ensure_ascii=False, indent=1, default=float)
with open(os.path.join(CARPETA_E1, "resumen_para_chat.txt"), "w", encoding="utf-8") as f:
    for celda, lineas in SALIDA_TXT.items():
        f.write(f"\n##### {celda} #####\n" + "\n".join(lineas) + "\n")
say(f"Registro escrito en {CARPETA_E1}")
say("Pega esta salida en el chat (no trae RUC ni procesos uno por uno).")

In [ ]:
# =====================================================================================
# CUADERNO 20 (MOTOR) · CELDA M15. E2: nivel de rareza y número de variables alteradas (enmienda D, 5.4)
#   Tipos nuevos de inyección. Cada variable alterada se lleva al cuantil q de su variable de origen dentro
#   del grupo (q = 0,95; 0,99 y 0,999; en plazo de ofertas y antigüedad el riesgo está en los valores bajos,
#   así que se usa el cuantil 1 - q), solo en procesos cuyo valor actual es menos extremo.
#     1 variable : precio | plazo de ofertas | antigüedad del proveedor | monto sobre capital
#     2 variables: precio + plazo | antigüedad + monto sobre capital
#     4 variables: las cuatro juntas
#   Solo cambia el percentil de las variables alteradas: las demás (incluidas debutante, cuota, recurrencia
#   y recompra) quedan como están. Representación normal_sup. Esta celda no ajusta modelos.
#   Antes de correrla: pegar M15, M16 y M17 en el cuaderno y guardarlo (Ctrl+S); M1 a M8 ya corridos.
# =====================================================================================
empezar("M15")
t_celda = time.time()
faltan = [n for n in ["ANIOS", "PLAN", "REGLAS", "DF_TODO", "df", "X", "recalcular", "valores_orientados", "clave_grupo",
                      "P5_PLAZO", "P99_CAPITAL", "AN_CONF", "correr", "una_semilla", "cambios_vigentes", "REPRODUCE",
                      "huella_codigo_cuaderno"] if n not in globals()]
if faltan:
    raise SystemExit("Faltan resultados de M1 a M8 en esta sesión (" + ", ".join(faltan) + "): corre M1 a M8 y vuelve a M15.")
if not REPRODUCE:
    raise SystemExit("El control de la celda M8 no reprodujo H1: no se corre E2.")

# --- el código de M1 a M14 es el de E0, E3b y E1; y el cuaderno guardado ya trae M15, M16 y M17 ---
HUELLA_14_ESPERADA = "b11503832e952417"
def partes_del_cuaderno():
    nb = json.load(open(CUADERNO, encoding="utf-8"))
    partes = []
    for c in nb["cells"]:
        if c["cell_type"] != "code":
            continue
        src = c["source"] if isinstance(c["source"], str) else "".join(c["source"])
        if any(l.startswith("# CUADERNO 20 (MOTOR)") for l in src.splitlines()[:3]):
            partes.append(src)
    return partes
def huella_de(partes):
    return hashlib.sha256("\n# ----- fin de celda -----\n".join(partes).encode("utf-8")).hexdigest()
PARTES = partes_del_cuaderno()
if len(PARTES) < 17:
    raise SystemExit(f"El cuaderno guardado tiene {len(PARTES)} celdas del motor y deben ser 17 o más: "
                     "pega M15, M16 y M17, guarda con Ctrl+S y vuelve a correr M15.")
if not huella_de(PARTES[:14]).startswith(HUELLA_14_ESPERADA):
    raise SystemExit("Las celdas M1 a M14 del cuaderno ya no son las de E0, E3b y E1: para y avisa.")
HUELLA_CODIGO_E2 = huella_de(PARTES)
say(f"Celdas M1 a M14 iguales a las de E0, E3b y E1 ({HUELLA_14_ESPERADA}) | código del cuaderno con E2 "
    f"{HUELLA_CODIGO_E2[:16]} ({len(PARTES)} celdas)")

# --- definición de los tipos ---
VARS_E2 = {"precio":  ("precio_alto",     "pct_precio_alto"),
           "plazo":   ("plazo_corto",     "pct_plazo_corto"),
           "antig":   ("proveedor_nuevo", "pct_proveedor_nuevo"),
           "capital": ("monto_capital",   "pct_monto_capital")}
for corto, (var, pct) in VARS_E2.items():
    if pct not in REGLAS:
        raise SystemExit(f"No encuentro la regla de {pct} en reproduccion_indicadores.json: para y avisa.")
COMBOS_E2 = [["precio"], ["plazo"], ["antig"], ["capital"], ["precio", "plazo"], ["antig", "capital"],
             ["precio", "plazo", "antig", "capital"]]
CUANTILES_E2 = [0.95, 0.99, 0.999]
ETQ_Q = {0.95: "95", 0.99: "99", 0.999: "999"}
TIPOS_E2 = {f"q{ETQ_Q[q]}_{'+'.join(c)}": (q, c) for q in CUANTILES_E2 for c in COMBOS_E2}
say(f"{len(TIPOS_E2)} tipos nuevos: {len(CUANTILES_E2)} cuantiles x {len(COMBOS_E2)} combinaciones "
    f"(1 variable: 4, 2 variables: 2, 4 variables: 1)")
say("\nRegla de cada variable alterada (la misma de la base):")
say(pd.DataFrame([dict(variable=c, percentil=p, fuente=REGLAS[p]["fuente"], signo=REGLAS[p]["signo"],
                       agrupacion=REGLAS[p]["agrupacion"], subconjunto=REGLAS[p]["subconjunto"])
                  for c, (_, p) in VARS_E2.items()]).to_string(index=False))

# --- valor de destino: cuantil q (de riesgo) de la variable de origen en cada grupo de la regla ---
def objetivo_cuantil(pct, q):
    """Valor de origen que corresponde al cuantil q de riesgo: q alto = más extremo (signo x fuente)."""
    regla = REGLAS[pct]
    v = valores_orientados(DF_TODO, regla)
    k = clave_grupo(DF_TODO, regla["agrupacion"])
    return regla["signo"] * v.groupby(k).quantile(q)
OBJETIVO_E2 = {(p, q): objetivo_cuantil(p, q) for _, p in VARS_E2.values() for q in CUANTILES_E2}

def coincide(a, b):
    a, b = a.dropna(), b.dropna()
    comunes = a.index.intersection(b.index)
    return len(comunes) > 0 and len(comunes) == len(a) == len(b) and bool(np.allclose(a.loc[comunes], b.loc[comunes], rtol=1e-9, atol=1e-9))
c1 = coincide(OBJETIVO_E2[("pct_plazo_corto", 0.95)], P5_PLAZO)
c2 = coincide(OBJETIVO_E2[("pct_monto_capital", 0.99)], P99_CAPITAL)
say(f"\nControl de los destinos: plazo con q = 0,95 coincide con el percentil 5 de H1: {'sí' if c1 else 'NO'}; "
    f"monto sobre capital con q = 0,99 coincide con el percentil 99 de H1: {'sí' if c2 else 'NO'}")
if not (c1 and c2):
    raise SystemExit("El destino por cuantil no coincide con las constantes de H1: pégame esta salida.")

# --- recetas: procesos elegibles y percentiles nuevos de cada tipo ---
def recetas_e2(plan, filas_a):
    sub = df.loc[filas_a]
    usar = plan["usar"]
    elegible, destino = {}, {}
    for corto, (var, pct) in VARS_E2.items():
        if var not in usar:
            continue
        regla = REGLAS[pct]
        clave = clave_grupo(sub, regla["agrupacion"])
        actual = valores_orientados(sub, regla)                     # valor alto = más riesgo
        for q in CUANTILES_E2:
            obj = clave.map(OBJETIVO_E2[(pct, q)])
            elegible[(corto, q)] = (actual.notna() & obj.notna() & (actual < regla["signo"] * obj)).to_numpy()
            destino[(corto, q)] = obj.to_numpy(dtype=float)
    R = {}
    for tipo, (q, combo) in TIPOS_E2.items():
        if any((c, q) not in elegible for c in combo):
            continue                                                # el grupo no usa alguna de esas variables
        ok = np.logical_and.reduce([elegible[(c, q)] for c in combo])
        filas = sub.index[ok]
        if len(filas) == 0:
            continue
        cambios = {}
        for c in combo:
            var, pct = VARS_E2[c]
            cambios[var] = recalcular(pct, filas, destino[(c, q)][ok])
        R[tipo] = (filas, cambios)
    return R

ANIOS_E2 = {}
for anio in ANIOS:                                                  # 2024 y los años de confirmación usados
    recetas = {}
    for g in ANIOS[anio]["grupos"]:
        plan = PLAN[g]
        filas_a = plan["idx"][(df.loc[plan["idx"], "anio_proc"] == anio).to_numpy()]
        recetas[g] = recetas_e2(plan, filas_a)
    ANIOS_E2[anio] = dict(grupos=ANIOS[anio]["grupos"], recetas=recetas)

filas = []
for anio, A in ANIOS_E2.items():
    for tipo in TIPOS_E2:
        gs = [g for g in A["grupos"] if tipo in A["recetas"][g]]
        filas.append(dict(anio=anio, tipo=tipo, grupos=len(gs), elegibles=sum(len(A["recetas"][g][tipo][0]) for g in gs)))
TAB_E2 = pd.DataFrame(filas).pivot(index="tipo", columns="anio", values=["grupos", "elegibles"]).reindex(list(TIPOS_E2))
say("\nGrupos que admiten cada tipo y procesos elegibles (suma de los grupos), por año:")
say(TAB_E2.to_string())
vacios = [(a, t) for a, A in ANIOS_E2.items() for t in TIPOS_E2 if not any(t in A["recetas"][g] for g in A["grupos"])]
if vacios:
    raise SystemExit(f"Hay tipos sin ningún proceso elegible en algún año: {vacios[:5]}. Pégame esta salida.")

# --- controles de orientación y de nivel: el percentil nuevo debe ser mayor que el actual y cercano a q ---
acum = {}
for anio, A in ANIOS_E2.items():
    for g, R in A["recetas"].items():
        for tipo, (filas_t, cambios) in R.items():
            q, combo = TIPOS_E2[tipo]
            for corto in combo:
                var, pct = VARS_E2[corto]
                nuevo = np.asarray(cambios[var], dtype=float)
                actual = num(df.loc[filas_t, pct]).to_numpy()
                acum.setdefault((corto, q), []).append(np.column_stack([nuevo, actual]))
filas = []
for (corto, q), partes in acum.items():
    M = np.vstack(partes)
    con = np.isfinite(M[:, 0]) & np.isfinite(M[:, 1])
    filas.append(dict(variable=corto, q=q, filas=int(len(M)), percentil_nuevo_medio=round(float(np.nanmean(M[:, 0])), 4),
                      percentil_actual_medio=round(float(np.nanmean(M[:, 1])), 4),
                      pct_nuevo_mayor_que_actual=round(100 * float((M[con, 0] > M[con, 1]).mean()), 3)))
CTRL_E2 = pd.DataFrame(filas).sort_values(["variable", "q"]).reset_index(drop=True)
say("\nControl de las recetas (percentil de la base antes y después de la inyección):")
say(CTRL_E2.to_string(index=False))
if (CTRL_E2["pct_nuevo_mayor_que_actual"] < 99.9).any():
    raise SystemExit("Hay inyecciones que no suben el percentil: la orientación o el destino fallan. Pégame esta salida.")
lejos = CTRL_E2[(CTRL_E2["percentil_nuevo_medio"] - CTRL_E2["q"]).abs() > 0.02]
if len(lejos):
    ALERTAS.append("E2: el percentil alcanzado difiere del nominal en más de 0,02 (empates en la variable): " +
                   ", ".join(f"{r.variable} q={r.q}" for r in lejos.itertuples()))
    say("Aviso: en estas variables el percentil alcanzado se aparta de q por los empates de la variable de origen: " +
        ", ".join(f"{r.variable} q={r.q}" for r in lejos.itertuples()))
# --- ¿los tres cuantiles dan tres valores distintos? Con muchos empates (días enteros) q = 0,99 y q = 0,999 pueden ser el mismo valor ---
grupos_e2 = sorted({g for A in ANIOS_E2.values() for g in A["grupos"]})
filas = []
for corto, (var, pct) in VARS_E2.items():
    regla = REGLAS[pct]
    n_g, ig_a, ig_b = 0, 0, 0
    for g in grupos_e2:
        if var not in PLAN[g]["usar"]:
            continue
        k = clave_grupo(df.loc[PLAN[g]["idx"][:1]], regla["agrupacion"]).iloc[0]
        d = [OBJETIVO_E2[(pct, q)].get(k, np.nan) for q in CUANTILES_E2]
        if np.isnan(d).any():
            continue
        n_g += 1
        ig_a += int(d[0] == d[1])
        ig_b += int(d[1] == d[2])
    filas.append(dict(variable=corto, grupos=n_g, destino_q095_igual_q099=ig_a, destino_q099_igual_q0999=ig_b))
DESTINOS_E2 = pd.DataFrame(filas)
say("\nGrupos en que dos cuantiles consecutivos dan el mismo valor de destino (el tipo sería el mismo):")
say(DESTINOS_E2.to_string(index=False))
if (DESTINOS_E2[["destino_q095_igual_q099", "destino_q099_igual_q0999"]].to_numpy() > 0).any():
    ALERTAS.append("E2: en algunos grupos dos cuantiles consecutivos tienen el mismo destino (empates en la variable de origen)")
say(f"[M15: {time.time() - t_celda:.0f} s]")

In [ ]:
# =====================================================================================
# CUADERNO 20 (MOTOR) · CELDA M16. E2: correr las inyecciones por cuantil y número de variables (enmienda D, 5.4)
#   Usa exactamente las funciones del motor (celda M8: una_semilla y correr); solo cambia el registro de
#   recetas (los 21 tipos de la celda M15). Mismo reparto de inyecciones que H1: el 1 % de los procesos de
#   cada año y grupo, repartido entre los tipos. Los cinco métodos, representación normal_sup.
#   2024 con las semillas 1 a 20 y los años de confirmación con las semillas 21 a 40 (60 semillas en total).
#   Unas 2 horas. Guarda el avance tras cada semilla: si se corta, pon en CARPETA_E2 la carpeta que ya
#   existe (por ejemplo r"C:\SERCOP_FINAL\modelos\20261123_0930_expD_E2") y vuelve a correr M16.
# =====================================================================================
empezar("M16")
if not REPRODUCE:
    raise SystemExit("El control de la celda M8 no reprodujo H1: no se corre E2.")
if "ANIOS_E2" not in globals() or "HUELLA_CODIGO_E2" not in globals():
    raise SystemExit("Falta correr M15 en esta sesión.")
CARPETA_E2 = None

def correr_e2(carpeta):
    """Corre el motor con las recetas de E2: cambia el registro ANIOS solo mientras dura la corrida."""
    global ANIOS
    original = ANIOS
    ANIOS = ANIOS_E2
    try:
        plan_e2 = {a: (SEMILLAS_EXPLORACION if a == 2024 else SEMILLAS_CONFIRMACION) for a in ANIOS_E2}
        return correr("E2", plan_e2, set(TIPOS_E2), {"normal_sup": (X, cambios_vigentes)}, carpeta=carpeta)
    finally:
        ANIOS = original

CARPETA_E2, E2R = correr_e2(CARPETA_E2)
if ANIOS is ANIOS_E2:
    raise SystemExit("El registro de recetas de H1 no se restituyó: no sigas y avisa.")
say(f"E2 terminado: {len(E2R):,} filas en {CARPETA_E2}")

In [ ]:
# =====================================================================================
# CUADERNO 20 (MOTOR) · CELDA M17. E2: curva de recuperación por cuantil y número de variables (enmienda D, 5.4)
#   Descriptivo, sin contraste ni regla de confirmación. Recuperación = proporción de inyecciones que quedan
#   en el 1 % superior del año en su grupo, por método, año, cuantil q y número de variables alteradas.
#   Los intervalos (95 %, remuestreo de semillas, 2.000 réplicas) son informativos. Se informa si el patrón de
#   2024 (exploración) se repite en los años de confirmación. El azar da 0,01.
# =====================================================================================
empezar("M17")
t_celda = time.time()
if "HUELLA_CODIGO_E2" not in globals():
    raise SystemExit("Falta correr M15 en esta sesión.")
if "CARPETA_E2" in globals() and CARPETA_E2:
    CARPETA_E2_REAL = CARPETA_E2
else:
    cands = sorted(d for d in os.listdir(SALIDA) if d.endswith("_expD_E2"))
    if not cands:
        raise SystemExit("No encuentro ninguna carpeta *_expD_E2 en modelos: corre M16.")
    CARPETA_E2_REAL = os.path.join(SALIDA, cands[-1])
R2 = pd.read_csv(os.path.join(CARPETA_E2_REAL, "E2_inyecciones.csv"))
R2["q"] = R2["tipo"].str.split("_").str[0].map({"q95": 0.95, "q99": 0.99, "q999": 0.999})
R2["combo"] = R2["tipo"].str.split("_", n=1).str[1]
R2["n_vars"] = R2["combo"].str.count(r"\+") + 1
if R2["q"].isna().any():
    raise SystemExit("Hay tipos con un nombre que no es de E2 en el archivo: revisa la carpeta.")
METODOS_E2 = ["ensamble", "base", "if", "lof", "ae"]
Q_ETQ = {0.95: "0,95", 0.99: "0,99", 0.999: "0,999"}
anios_hay = sorted(int(a) for a in R2["anio"].unique())
anios_esperados = sorted([2024] + list(AN_CONF))
say(f"Carpeta: {CARPETA_E2_REAL}")
say(f"Filas: {len(R2):,} | años {anios_hay} (esperados {anios_esperados}) | "
    f"semillas por año: {R2.groupby('anio')['semilla'].nunique().to_dict()} | tipos distintos: {R2['tipo'].nunique()}")
if anios_hay != anios_esperados:
    ALERTAS.append(f"E2: años en el archivo {anios_hay}, esperados {anios_esperados}")
cuenta_m = R2.groupby("metodo").size()
if cuenta_m.nunique() != 1 or set(cuenta_m.index) != set(METODOS_E2):
    raise SystemExit(f"Los cinco métodos no tienen las mismas inyecciones: {cuenta_m.to_dict()}")
iny = R2[R2["metodo"] == "ensamble"].groupby(["anio", "tipo"]).size().unstack(0)
iny = iny.reindex(list(TIPOS_E2)).fillna(0).astype(int)
say("\nInyecciones por tipo y año (mínimo, mediana, máximo entre los 21 tipos):")
say(iny.agg(["min", "median", "max"]).T.to_string())
if (iny == 0).any().any():
    ALERTAS.append("E2: hay tipos sin inyecciones en algún año")

def media_ic(sub, valor):
    """Recuperación media con IC 95 % por remuestreo de semillas (las semillas son la unidad)."""
    g = sub.groupby("semilla")[valor].agg(["sum", "count"]).astype(float)
    if g["count"].sum() == 0:
        return dict(inyecciones=0, semillas=0, rec=np.nan, ic95_inf=np.nan, ic95_sup=np.nan)
    s, c = g["sum"].to_numpy(), g["count"].to_numpy()
    idx = np.random.default_rng(SEMILLA).integers(0, len(s), size=(B_REMUESTREO, len(s)))
    with np.errstate(invalid="ignore", divide="ignore"):
        boot = s[idx].sum(axis=1) / c[idx].sum(axis=1)
    ic = np.nanpercentile(boot, [2.5, 97.5])
    return dict(inyecciones=int(c.sum()), semillas=int(len(s)), rec=float(s.sum() / c.sum()),
                ic95_inf=float(ic[0]), ic95_sup=float(ic[1]))

filas = []
for corte in [1, 2, 5]:
    valor = f"en_top_{corte}"
    for anio in anios_hay:
        for metodo in METODOS_E2:
            Rm = R2[(R2["anio"] == anio) & (R2["metodo"] == metodo)]
            def agrega(nivel, sub, q_etq, n_etq, tipo=""):
                filas.append(dict(corte=corte, anio=int(anio), metodo=metodo, nivel=nivel, q_etq=q_etq, n_etq=n_etq,
                                  tipo=tipo, **media_ic(sub, valor)))
            agrega("total", Rm, "todos", "todas")
            for q, qe in Q_ETQ.items():
                agrega("por_q", Rm[Rm["q"] == q], qe, "todas")
            for n in [1, 2, 4]:
                agrega("por_n_vars", Rm[Rm["n_vars"] == n], "todos", str(n))
                for q, qe in Q_ETQ.items():
                    agrega("celda", Rm[(Rm["n_vars"] == n) & (Rm["q"] == q)], qe, str(n))
            for tipo in TIPOS_E2:
                q_t, _ = TIPOS_E2[tipo]
                agrega("tipo", Rm[Rm["tipo"] == tipo], Q_ETQ[q_t], str(tipo.count("+") + 1), tipo)
LARGA = pd.DataFrame(filas)
L1 = LARGA[LARGA["corte"] == 1]
ORD_N = {"1": 0, "2": 1, "4": 2, "todas": 3}
ORD_Q = ["0,95", "0,99", "0,999", "todos"]

say("\nRECUPERACIÓN AL 1 % POR AÑO, MÉTODO, NÚMERO DE VARIABLES (filas) Y CUANTIL q (columnas). Azar = 0,01.")
say("(la fila 'todas' agrupa los tres números de variables; la columna 'todos' agrupa los tres cuantiles;")
say(" cada tipo pesa igual, así que 'todas' cuenta 4 tipos de una variable, 2 de dos y 1 de cuatro)")
for anio in anios_hay:
    T = L1[(L1["anio"] == anio) & (L1["nivel"] != "tipo")].copy()
    T["ord_n"] = T["n_etq"].map(ORD_N)
    P = T.pivot_table(index=["metodo", "ord_n", "n_etq"], columns="q_etq", values="rec").reindex(columns=ORD_Q)
    P = P.reset_index().assign(o=lambda d: d["metodo"].map({m: i for i, m in enumerate(METODOS_E2)})) \
         .sort_values(["o", "ord_n"]).drop(columns=["o", "ord_n"]).set_index(["metodo", "n_etq"]).round(3)
    P.index.names = ["metodo", "variables"]
    say(f"\nAño {anio}:")
    say(P.to_string())

def texto_ic(r):
    return f"{r.rec:.3f} [{r.ic95_inf:.3f}-{r.ic95_sup:.3f}]"
for titulo, nivel, cols, col_de in [("POR CUANTIL q (los tres números de variables juntos), con IC 95 %", "por_q", ["0,95", "0,99", "0,999"], "q_etq"),
                                    ("POR NÚMERO DE VARIABLES (los tres cuantiles juntos), con IC 95 %", "por_n_vars", ["1", "2", "4"], "n_etq")]:
    T = L1[L1["nivel"] == nivel].copy()
    T["txt"] = [texto_ic(r) for r in T.itertuples()]
    P = T.pivot_table(index=["anio", "metodo"], columns=col_de, values="txt", aggfunc="first")[cols]
    P = P.reset_index().assign(o=lambda d: d["metodo"].map({m: i for i, m in enumerate(METODOS_E2)})) \
         .sort_values(["anio", "o"], ascending=[False, True]).drop(columns="o").set_index(["anio", "metodo"])
    say(f"\n{titulo}:")
    say(P.to_string())

T = L1[L1["nivel"] == "tipo"].copy()
T["combo"] = T["tipo"].str.split("_", n=1).str[1]
P = T.pivot_table(index=["metodo", "combo"], columns=["anio", "q_etq"], values="rec")
P = P.reindex(columns=[(a, q) for a in sorted(anios_hay, reverse=True) for q in ["0,95", "0,99", "0,999"]])
orden_combo = {"+".join(c): i for i, c in enumerate(COMBOS_E2)}
P = P.reset_index().assign(o=lambda d: d["metodo"].map({m: i for i, m in enumerate(METODOS_E2)}),
                           oc=lambda d: d["combo"].map(orden_combo)).sort_values(["o", "oc"]).drop(columns=["o", "oc"]) \
     .set_index(["metodo", "combo"]).round(3)
say("\nRECUPERACIÓN AL 1 % POR COMBINACIÓN DE VARIABLES (filas), AÑO Y CUANTIL (columnas):")
say(P.to_string())

# --- ¿el patrón de 2024 se repite en los años de confirmación? (descriptivo) ---
def vec(anio, metodo, nivel, etiquetas, col):
    s = L1[(L1["anio"] == anio) & (L1["metodo"] == metodo) & (L1["nivel"] == nivel)].set_index(col)["rec"]
    return [float(s[e]) for e in etiquetas]
filas_p = []
for metodo in METODOS_E2:
    base24 = L1[(L1["anio"] == 2024) & (L1["metodo"] == metodo) & (L1["nivel"] == "tipo")].set_index("tipo")["rec"].reindex(list(TIPOS_E2))
    celdas24 = L1[(L1["anio"] == 2024) & (L1["metodo"] == metodo) & (L1["nivel"] == "celda")].set_index(["n_etq", "q_etq"])["rec"]
    for anio in sorted(anios_hay, reverse=True):
        vq = vec(anio, metodo, "por_q", ["0,95", "0,99", "0,999"], "q_etq")
        vn = vec(anio, metodo, "por_n_vars", ["1", "2", "4"], "n_etq")
        fila = dict(metodo=metodo, anio=anio, sube_con_q=bool(np.all(np.diff(vq) >= 0)), sube_con_n_vars=bool(np.all(np.diff(vn) >= 0)),
                    rec_q_095=round(vq[0], 3), rec_q_0999=round(vq[2], 3), rec_1_var=round(vn[0], 3), rec_4_vars=round(vn[2], 3))
        if anio != 2024:
            este = L1[(L1["anio"] == anio) & (L1["metodo"] == metodo) & (L1["nivel"] == "tipo")].set_index("tipo")["rec"].reindex(list(TIPOS_E2))
            fila["rho_21_tipos_con_2024"] = round(float(spearmanr(base24, este).correlation), 3)
            c_a = L1[(L1["anio"] == anio) & (L1["metodo"] == metodo) & (L1["nivel"] == "celda")].set_index(["n_etq", "q_etq"])["rec"]
            fila["mayor_diferencia_rejilla_con_2024"] = round(float((c_a - celdas24).abs().max()), 3)
        filas_p.append(fila)
PATRON = pd.DataFrame(filas_p)
say("\nPATRÓN POR AÑO (descriptivo; 2024 es la exploración, los otros años dicen si se repite):")
say("sube_con_q: la recuperación no baja de q 0,95 a 0,99 a 0,999 | sube_con_n_vars: no baja de 1 a 2 a 4 variables")
say("rho: correlación de Spearman entre las 21 recuperaciones por tipo del año y las de 2024 | mayor_diferencia: en las 9 celdas q x variables")
say(PATRON.to_string(index=False))

# --- archivos de la corrida ---
LARGA.to_csv(os.path.join(CARPETA_E2_REAL, "E2_recuperacion.csv"), index=False)
iny.to_csv(os.path.join(CARPETA_E2_REAL, "E2_inyecciones_por_tipo_y_anio.csv"))
PATRON.to_csv(os.path.join(CARPETA_E2_REAL, "E2_patron.csv"), index=False)
CTRL_E2.to_csv(os.path.join(CARPETA_E2_REAL, "E2_control_recetas.csv"), index=False)
DESTINOS_E2.to_csv(os.path.join(CARPETA_E2_REAL, "E2_destinos_iguales.csv"), index=False)
TAB_E2.to_csv(os.path.join(CARPETA_E2_REAL, "E2_tipos_elegibles.csv"))
def a_registros(tabla):
    """Filas como diccionarios; los vacíos (NaN) pasan a null para que el JSON sea válido."""
    return json.loads(tabla.to_json(orient="records"))
reg = dict(analisis="E2", estatus="exploratorio descriptivo, enmienda D 5.4", enmienda_d=HUELLA_D, huella_codigo=HUELLA_CODIGO_E2,
           huella_celdas_m1_a_m14=HUELLA_14_ESPERADA, preregistro=FIRMA["huella_documento"], anios=anios_hay,
           semillas={int(a): sorted(int(s) for s in R2[R2["anio"] == a]["semilla"].unique()) for a in anios_hay},
           tipos=list(TIPOS_E2), control_recetas=a_registros(CTRL_E2), destinos_iguales=a_registros(DESTINOS_E2), filas=int(len(R2)),
           recuperacion_1pct=a_registros(L1[L1["nivel"] != "tipo"][["anio", "metodo", "nivel", "q_etq", "n_etq", "inyecciones", "rec", "ic95_inf", "ic95_sup"]]),
           patron=a_registros(PATRON), alertas=ALERTAS, versiones=VERSIONES,
           fecha=datetime.now().isoformat(timespec="seconds"), carpeta=CARPETA_E2_REAL)
with open(os.path.join(CARPETA_E2_REAL, "registro_E2.json"), "w", encoding="utf-8") as f:
    json.dump(reg, f, ensure_ascii=False, indent=1, default=float)
with open(os.path.join(CARPETA_E2_REAL, "resumen_para_chat.txt"), "w", encoding="utf-8") as f:
    for celda in ["M15", "M16", "M17"]:
        if celda in SALIDA_TXT:
            f.write(f"\n##### {celda} #####\n" + "\n".join(SALIDA_TXT[celda]) + "\n")
say(f"\nRegistro escrito en {CARPETA_E2_REAL}")
say("Pega esta salida en el chat (no trae RUC ni procesos uno por uno).")